# MACE from first principles

A Pass 3 derivation-and-implementation notebook.

**Target reader:** a machine-learning practitioner who understands tensors, linear layers, gradients, and ordinary message passing, but may know nothing about geometric deep learning.

**Scope:** the notebook contains reference explanations, transparent equation labs, and one complete educational MACE block. It is designed for inspection and learning, not as production training code; the final section states exactly which full-scale extensions remain outside this notebook.

## 0. Setup — imports

In [ ]:
from pathlib import Path

from ase.io import read
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from e3nn import o3

## The concrete problem — 3BPA

Before studying symmetry machinery, look at the real prediction problem. The 3BPA training file contains **500 snapshots of the same 27-atom molecule at different 300 K geometries**. A snapshot is also called a *frame*. It is one instantaneous arrangement of all 27 nuclei, not a different molecular species.

For every frame, DFT supplies two connected labels:

- one scalar total energy $E$ for the complete geometry;
- one three-dimensional force vector for each atom, $\mathbf F_a=-\nabla_{\mathbf r_a}E$.

The atomic identities $Z$ and positions $R$ are the input. MACE learns a scalar energy surface $E_\theta(R,Z)$; differentiating that learned energy with respect to every atomic position produces the predicted forces. The red vectors below therefore show the local direction and strength of the energy slope at one sampled geometry. They are scaled only for display.

The 300 K, 600 K, 1200 K, and dihedral test files still contain 3BPA. They test whether the learned energy surface remains accurate for geometries sampled under increasingly different conditions. Later we switch to methane only as a small, controlled geometry for understanding rotations and equivariance—it is not our benchmark molecule.

In [ ]:
DATA_PATH = Path.cwd() / "data/dataset_3BPA/train_300K.xyz"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data/dataset_3BPA/train_300K.xyz"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "papers/15-MACE/data/dataset_3BPA/train_300K.xyz"

three_bpa_frames = read(DATA_PATH, index=":", format="extxyz")
assert isinstance(three_bpa_frames, list)
assert len(three_bpa_frames) == 500
print(f"Loaded {len(three_bpa_frames)} frames from {DATA_PATH.name}")

In [ ]:
def show_random_3bpa_frame(frames, *, seed=42, force_scale=0.35):
    """Plot one reproducibly sampled ASE frame and its force vectors."""
    rng = np.random.default_rng(seed)
    frame_index = int(rng.integers(len(frames)))
    atoms = frames[frame_index]
    symbols = atoms.get_chemical_symbols()
    positions = np.asarray(atoms.positions, dtype=float)
    forces = np.asarray(atoms.get_forces(), dtype=float)
    energy = float(atoms.get_potential_energy())
    dihedrals = atoms.info.get("dihedrals")

    assert len(symbols) == 27
    assert positions.shape == forces.shape == (27, 3)

    force_norms = np.linalg.norm(forces, axis=1)
    element_colors = {"C": "#6c5ce7", "H": "#00bfa6", "N": "#2f80ed", "O": "#f2994a"}
    marker_colors = [element_colors.get(symbol, "#999999") for symbol in symbols]
    customdata = np.column_stack([np.arange(len(symbols)), np.asarray(symbols), forces, force_norms])

    atom_trace = go.Scatter3d(
        x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
        mode="markers",
        marker={"size": 7, "color": marker_colors, "line": {"width": 1, "color": "#222222"}},
        customdata=customdata,
        hovertemplate=(
            "atom %{customdata[0]} (%{customdata[1]})<br>"
            "position: (%{x:.3f}, %{y:.3f}, %{z:.3f}) Å<br>"
            "force: (%{customdata[2]:.3f}, %{customdata[3]:.3f}, %{customdata[4]:.3f}) eV/Å<br>"
            "|force|: %{customdata[5]:.3f} eV/Å<extra></extra>"
        ),
        name="atoms",
    )

    line_x, line_y, line_z = [], [], []
    for position, force in zip(positions, forces):
        line_x.extend([position[0], position[0] + force_scale * force[0], None])
        line_y.extend([position[1], position[1] + force_scale * force[1], None])
        line_z.extend([position[2], position[2] + force_scale * force[2], None])

    force_trace = go.Scatter3d(
        x=line_x, y=line_y, z=line_z, mode="lines",
        line={"color": "crimson", "width": 4},
        hoverinfo="skip", name="forces (scaled for display)",
    )
    title = f"3BPA frame {frame_index} | energy={energy:.6f} eV"
    if dihedrals is not None:
        title += f" | dihedrals={dihedrals}"

    figure = go.Figure([atom_trace, force_trace])
    for element, color in element_colors.items():
        figure.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None], mode="markers",
            marker={"size": 7, "color": color}, name=element,
            hoverinfo="skip", showlegend=True,
        ))
    figure.update_layout(
        title=title,
        scene={"xaxis_title": "x (Å)", "yaxis_title": "y (Å)", "zaxis_title": "z (Å)"},
        scene_aspectmode="data",
        margin={"l": 0, "r": 0, "b": 0, "t": 50},
    )
    figure.show()
    return frame_index, figure


sampled_frame_index, three_bpa_figure = show_random_3bpa_frame(three_bpa_frames, seed=42)

## Learning route

For every topic: write plain English → connect it to familiar ML → record shapes → code a tiny example → graph it → test the mathematical identity → draft the blog paragraph.

- [ ] atomic geometry and symmetry
- [ ] rotations, invariance, and equivariance
- [ ] Cartesian tensors and their rotation laws
- [ ] decomposition into independent rotational blocks
- [ ] representations, irreps, `l`, and `m`
- [ ] spherical harmonics as Fourier-like modes on the sphere
- [ ] Wigner-D matrices and numerical equivariance
- [ ] channels versus angular components
- [ ] tensor products and Clebsch–Gordan decomposition
- [ ] higher-order coupling, `ν`, and `η_ν`
- [ ] radial basis functions and cutoffs
- [ ] MACE Eqs. 8–11
- [ ] one minimal message block and symmetry tests

**Pedagogical reference:** Sections 5.1–5.3 of *A Hitchhiker's Guide to Geometric GNNs for 3D Atomic Systems*. The prompts also preserve useful questions and analogies from your Claude discussion, but the paper and numerical checks remain the authority.

## 1. Notation ledger — write this yourself

Create a table with one row for each symbol below. Leave no symbol undefined in your eventual blog explanation.

`i`, `j`, `k`, `k̃`, `l`, `m`, `ξ`, `ν`, `L`, `M`, `η_ν`

| Symbol | Meaning |
|---|---|
| `i` | Central / receiving atom |
| `j` | Neighbor atom of `i` |
| `k` | Output feature channel of `A` or `B` |
| `k̃` | Input feature channel being mixed into `k` |
| `l` | Angular-momentum / irrep order |
| `m` | Component within order `l`; `m = -l, ..., +l` |
| `ξ` | Tensor-product factor index; one of the `ν` factors |
| `ν` | Number of `A` factors in the tensor product |
| `L` | Output angular-momentum / irrep order of `B` |
| `M` | Component within output order `L` |
| `η_ν` | Label for the coupling path that produces total order `L` |

### Index warm-up — your explanation

Answer these in your own words:

- If `l = 1`, which `m` values exist?
- For `ν = 2`, what are the two factor slots?
- What does `A[i, k̃, l_ξ, m_ξ]` select?

**Blog sentence:** $l$ names the rotation type, $m$ names a component inside that type, $k$ names a learned feature channel, $\xi$ selects one of the $\nu$ multiplied factors, and $\eta_\nu$ records the route used to reach final order $(L,M)$.

For `l = 1`, `m` takes the integer values from `-l` to `+l`: `m = -1, 0, +1`.

For `ν = 2`, there are two tensor-product factor slots: `ξ = 1` and `ξ = 2`. They contain the angular-index pairs `(l₁, m₁)` and `(l₂, m₂)`.

The entry `A[i, k̃, l_ξ, m_ξ]` selects one feature value: the feature at atom `i`, input channel `k̃`, angular order `l_ξ`, and component `m_ξ`.

## Authoring contract for a non-GDL reader

Before continuing, write down the standard every section must meet:

- define a term before abbreviating it;
- give one concrete geometric example before formal notation;
- connect the operation to something familiar from ML;
- state every tensor shape and what each axis means;
- show one visual example whenever the object has geometric meaning;
- test the claimed symmetry numerically;
- state the real/complex basis and normalization convention.

Your reader promise:

# Part I — geometric building blocks

The paper equations come later. First make each primitive visible and testable.

## 2. Atomic geometry

### Learning prompt (preserved)

Explain central atom `i`, neighbor `j`, relative displacement, distance, unit direction, translation invariance, and permutation-invariant neighbor pooling.

### ML bridge

Connect neighbor pooling to aggregation in an ordinary message-passing network.

### Geometry before symmetry

Let atom $i$ be the centre and atom $j$ one of its neighbours. Their absolute positions are $\mathbf r_i,\mathbf r_j\in\mathbb R^3$. The edge from $i$ to $j$ is described by three objects, in dependency order:

$$
\mathbf r_{ji}=\mathbf r_j-\mathbf r_i\in\mathbb R^3,\qquad
r_{ji}=\lVert\mathbf r_{ji}\rVert\in\mathbb R,\qquad
\hat{\mathbf r}_{ji}=\mathbf r_{ji}/r_{ji}\in\mathbb R^3.
$$

The displacement stores the full relative geometry, the distance stores only separation, and the unit direction stores only orientation. Translating every atom by the same vector cancels in the subtraction, so all three edge objects are translation invariant.

For one centre with $E_i$ neighbours, stack the edge objects as `displacements: (E_i, 3)`, `distances: (E_i,)`, and `directions: (E_i, 3)`. A message function produces one row per edge. Summing those rows over the neighbour axis gives a centre feature that is unchanged when the neighbour rows are permuted. This is the same aggregation idea as an ordinary message-passing network; later sections constrain what each feature is allowed to do under rotation.

> **Keep this distinction close:** distance answers *how far*; unit direction answers *which way*. MACE deliberately processes them on separate branches before combining them.


We consider a central carbon atom which has four hydrogen atoms. 

### Ideal tetrahedral coordinates for methane

We place the carbon atom at the origin and choose four hydrogen directions from alternating corners of a cube:

$$
(1,1,1),\quad
(1,-1,-1),\quad
(-1,1,-1),\quad
(-1,-1,1).
$$

To turn these into unit directions, divide each vector by its length:

$$
\hat{\mathbf r}
=
\frac{(\pm1,\pm1,\pm1)}{\sqrt{3}}.
$$

Any pair of these directions has dot product

$$
\hat{\mathbf r}_a \cdot \hat{\mathbf r}_b = -\frac{1}{3}.
$$

Therefore, the angle between any two C–H bonds is

$$
\theta
=
\cos^{-1}\left(-\frac{1}{3}\right)
\approx 109.47^\circ,
$$

which is the tetrahedral bond angle.

These are ideal synthetic coordinates for the geometry lab, not measured experimental coordinates.

In [ ]:
pos_C = (torch.tensor((0, 0, 0))).unsqueeze(0)
sqrt_3 = torch.sqrt(torch.tensor(3))
pos_H1 = (torch.tensor((1,1,1))/sqrt_3).unsqueeze(0)
pos_H2 = (torch.tensor((1,-1,-1))/sqrt_3).unsqueeze(0)
pos_H3 = (torch.tensor((-1,1,-1))/sqrt_3).unsqueeze(0)
pos_H4 = (torch.tensor((-1,-1,1))/sqrt_3).unsqueeze(0)
pos_H2.shape
pos_C.shape
positions = torch.concat([pos_C, pos_H1, pos_H2, pos_H3, pos_H4], dim=0)

In [ ]:
positions

In [ ]:
# Reusable interactive 3D molecular plot
from IPython.display import HTML, display

def plot_molecule_3d(positions, labels, title="Molecule geometry"):
    # Plot atoms and bonds from a central atom to every neighbor.
    xyz = positions.detach().cpu().numpy()
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("positions must have shape (number_of_atoms, 3)")
    if len(labels) != len(xyz):
        raise ValueError("labels must contain one entry per atom")

    fig = go.Figure()
    for neighbor_idx in range(1, len(xyz)):
        fig.add_trace(go.Scatter3d(
            x=[xyz[0, 0], xyz[neighbor_idx, 0]],
            y=[xyz[0, 1], xyz[neighbor_idx, 1]],
            z=[xyz[0, 2], xyz[neighbor_idx, 2]],
            mode="lines",
            line=dict(color="gray", width=6),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers+text",
        text=labels,
        textposition="top center",
        hovertemplate="%{text}<br>(%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>",
        marker=dict(
            size=[22] + [14] * (len(xyz) - 1),
            color=["black"] + ["lightgray"] * (len(xyz) - 1),
            line=dict(color="darkslategray", width=1),
        ),
        showlegend=False,
    ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=45),
    )
    return fig

fig = plot_molecule_3d(
    positions,
    labels=["C", "H1", "H2", "H3", "H4"],
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [ ]:
torch.manual_seed(42)
translation_vector = 5*torch.rand(1, 3)
new_positions = positions + translation_vector
neighbor_order = torch.randperm(4) + 1
indices = torch.cat([torch.tensor([0]), neighbor_order])
new_pos_perm = new_positions[indices]
labels=["C", "H1", "H2", "H3", "H4"]
new_labels_perm = [labels[i] for i in indices.tolist()]
# new_pos_perm
fig = plot_molecule_3d(
    new_pos_perm,
    labels=new_labels_perm,
    title="Translated methane with permuted neighbour rows",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

The cell above translates the whole molecule and permutes the neighbour rows; it does not rotate it. Subtracting the centre removes the translation, and summing edge messages will remove dependence on row order. Rotation is the separate experiment in the next section.

In [ ]:
print("original order:", labels)
print("permuted order:", new_labels_perm)
print("row indices:", indices.tolist())

In [ ]:
print(positions - positions[0])


In [ ]:
new_positions - new_positions[0]

In [ ]:
# LAB 2 — atomic geometry: complete shape and symmetry check.
relative_displacements = positions[1:] - positions[0]                 # (neighbors=4, xyz=3)
neighbor_distances = torch.linalg.vector_norm(relative_displacements, dim=-1)
neighbor_directions = relative_displacements / neighbor_distances[:, None]
translated_displacements = new_positions[1:] - new_positions[0]

original_pooled_displacement = relative_displacements.sum(dim=0)
permuted_neighbor_rows = new_pos_perm[1:] - new_pos_perm[0]
permuted_pooled_displacement = permuted_neighbor_rows.sum(dim=0)

assert relative_displacements.shape == (4, 3)
assert neighbor_distances.shape == (4,)
assert neighbor_directions.shape == (4, 3)
assert torch.allclose(neighbor_distances, torch.ones(4))
assert torch.allclose(torch.linalg.vector_norm(neighbor_directions, dim=-1), torch.ones(4))
assert torch.allclose(translated_displacements, relative_displacements)
assert torch.allclose(
    permuted_pooled_displacement, original_pooled_displacement, atol=1e-6, rtol=1e-6
)  # float32 subtraction after translation introduces roundoff near zero
print("displacement, distance, direction shapes:", relative_displacements.shape, neighbor_distances.shape, neighbor_directions.shape)
print("translation invariant and neighbour-sum permutation invariant: True")

## 3. Rotations, invariance, and equivariance

### Learning prompt (preserved)

Using one rotated vector, explain a rotation matrix, an invariant quantity, an equivariant quantity, and the two routes in an equivariance check.

### ML bridge

Compare equivariance with weight sharing in convolution, without claiming they are identical.

This section stays in ordinary Cartesian coordinates. It establishes the invariant/equivariant distinction and the two-route test before later sections introduce spherical harmonics or Wigner-$D$ matrices.

A three-dimensional rotation matrix $R$ maps a vector $\mathbf v$ to a rotated vector $\mathbf v'=R\mathbf v$. For a counter-clockwise rotation by angle $\theta$ about the $z$ axis,

$$
R_z(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0\\
\sin\theta & \cos\theta & 0\\
0 & 0 & 1
\end{bmatrix}.
$$

A scalar such as $\lVert\mathbf v\rVert$ is **invariant**: it has the same value before and after rotation. The vector itself is **equivariant**: it changes, but by the known rule $\mathbf v' = R\mathbf v$. Every later equivariance check compares two routes: transform the input and then run the operation, or run the operation and then transform its output.

In [ ]:
positions

In [ ]:
torch.tensor([[1,2],[3,4]])

In [ ]:
def rotate_z_axis(positions, theta):
    theta = torch.tensor(theta)
    c = torch.cos(theta)
    s = torch.sin(theta)
    rotation_matrix = torch.tensor(
        [
            [c, -s, 0],
            [s, c, 0],
            [0,0,1]
        ]
        )
    rotated_positions = positions @ rotation_matrix.T 
    return rotation_matrix, rotated_positions
rotation_matrix, rotated_positions = rotate_z_axis(positions, math.pi/2)

In [ ]:
print(positions)
print(rotated_positions)

In [ ]:
fig = plot_molecule_3d(
    rotated_positions,
    labels=labels,
    title="Ideal methane after a 90-degree z rotation",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [ ]:
print(rotation_matrix.T @ rotation_matrix)
print(torch.linalg.det(rotation_matrix))

$R^\top R=I$ means $R^{-1}=R^\top$. Therefore $(R\mathbf v)\cdot(R\mathbf w)=\mathbf v\cdot\mathbf w$: lengths and angles are preserved. The additional check $\det R=+1$ distinguishes a proper rotation from a reflection.

In [ ]:
rot_vec1, new_position_1 = rotate_z_axis(positions, math.pi/4)
rot_vec2, new_position_2 = rotate_z_axis(new_position_1, math.pi/3)
print(new_position_2)
new_position_3 = positions @ (rot_vec2 @ rot_vec1).T
print(new_position_3)


The cell above compares the resulting **positions**, not the rotation matrices themselves. Applying $R_1$ and then $R_2$ gives the same coordinates as applying the composed rotation $R_2R_1$ once. This is the concrete composition rule that later becomes $D^{(l)}(R_2R_1)=D^{(l)}(R_2)D^{(l)}(R_1)$ in feature space.

In [ ]:
# LAB 3 — rotations: invariant scalar, equivariant vector, and composition.
test_vector = positions[1]
rotated_test_vector = rotation_matrix @ test_vector
sequential_positions = new_position_2
composed_positions = positions @ (rot_vec2 @ rot_vec1).T

assert torch.allclose(rotation_matrix.T @ rotation_matrix, torch.eye(3), atol=1e-6)
assert torch.allclose(torch.linalg.det(rotation_matrix), torch.tensor(1.0), atol=1e-6)
assert torch.allclose(torch.linalg.vector_norm(rotated_test_vector), torch.linalg.vector_norm(test_vector))
assert torch.allclose(sequential_positions, composed_positions, atol=1e-6)
print("norm invariant:", torch.linalg.vector_norm(test_vector).item())
print("vector transforms by R:", rotated_test_vector)
print("sequential and composed rotations agree: True")

## 4. Cartesian tensors before irreps

Do not use the word *irrep* yet. Begin with objects an ML practitioner already knows: a scalar, a 3D vector, and a 3×3 matrix.

### Your explanation

Starting with two vectors `v` and `w`, explain the outer product `T = v ⊗ w`, why it has shape `(3, 3)`, and why it contains nine numbers rather than three. Then derive its rotation law from `v' = Rv` and `w' = Rw`.

### Shape and transformation ledger

Record the shape and rotation law of a scalar, a vector, and this matrix. Explicitly distinguish **tensor rank** from the number of stored components.

### ML bridge

Relate the outer product to multiplicative feature interactions, but explain why geometry dictates how the result must transform.

### Blog paragraph

Let us begin with the simplest operation that combines two vectors without throwing away their directional information. Consider

$$
v = (1,2,5), \qquad w = (6,7,17).
$$

The **outer product** (also called a rank-2 tensor product) forms every pairwise product between one component of `v` and one component of `w`. We write it as $T = v \otimes w$. Its entries are

$$
T_{ij} = v_i w_j,
$$

so two vectors with shape `(3,)` produce a matrix with shape `(3, 3)`, containing $3 \times 3 = 9$ values:

$$
T =
\begin{bmatrix}
v_1w_1 & v_1w_2 & v_1w_3 \\
v_2w_1 & v_2w_2 & v_2w_3 \\
v_3w_1 & v_3w_2 & v_3w_3
\end{bmatrix}.
$$

This is different from a dot product. The dot product compresses two vectors into one scalar, while the outer product preserves every pairwise interaction. That makes it useful when the relationship between directions matters, for example in second-order geometric features and multiplicative feature interactions in machine learning.

The operation itself has no learned parameters; it is just multiplication and arrangement. Its geometric importance comes from its transformation law. If both vectors are rotated by `R`, then

$$
(Rv) \otimes (Rw) = R T R^\top.
$$

Thus `T` is a Cartesian rank-2 tensor. The nine entries are not yet nine unrelated feature types: this nine-dimensional object is reducible and will later split into the `1 + 3 + 5` rotational blocks.

In [ ]:
v = torch.tensor([1., 2., 5.])
w = torch.tensor([6., 7., 17.])
T = torch.outer(v, w)
T

Next, we'll rotate each vector v and w individually and find the tensor product on the rotated vectors. We want to verify the following identity. 

\((Rv)\otimes(Rw)=RTR^\top.\)

Let's choose the rotation matrix for 45° around z axis. 
So, 
R = [cospi/4 -sinpi/4 0 

    sinpi/4.  cospi/4 0
    
    0 0 1]

In [ ]:
c = torch.cos(torch.tensor(torch.pi/4))
s = torch.sin(torch.tensor(torch.pi/4))
R = torch.tensor([[c,-s,0], [s,c,0], [0,0,1]])
rot_v = R @ (v.reshape(-1,1))
rot_w = R @ (w.reshape(-1,1))
rot_T = torch.outer(rot_v.reshape(-1), rot_w.reshape(-1))
print(rot_T)

print(R @ T @ R.T)




The outer product is equivariant: rotating both input vectors and then forming their outer product gives the same result as rotating the resulting rank-2 tensor with \(RTR^\top\).

In [ ]:
# LAB 4 — a Cartesian rank-2 tensor: two-route equivariance check.
T_from_vectors = torch.outer(v, w)
T_rotated_from_vectors = torch.outer(R @ v, R @ w)
T_then_rotated = R @ T_from_vectors @ R.T

assert T_from_vectors.shape == (3, 3)
assert T_from_vectors.flatten().shape == (9,)
assert torch.allclose(T_rotated_from_vectors, T_then_rotated, atol=1e-6)
print("T shape and flattened shape:", T_from_vectors.shape, T_from_vectors.flatten().shape)
print("outer-product equivariance error:", torch.max(torch.abs(T_rotated_from_vectors - T_then_rotated)).item())

## 5. Discover the independent rotational pieces

The nine matrix entries do not all behave as one indivisible object. Discover the smaller families before naming them.

### Your explanation

Split a generic 3×3 matrix into:

1. its trace part;
2. its antisymmetric part;
3. its symmetric-traceless part.

Count their independent numbers and explain, from the rotation experiment, what it means for each family to remain closed under rotation. Use the concrete statement `9 = 1 + 3 + 5` before introducing any angular-order notation.

### Naming checkpoint

Only after the experiment, write your own definition of an **irreducible representation**. Test the phrase *an independent rotational block*: its components may mix with one another, but rotation does not leak them into another block. Where is this analogy useful, and where might it be misleading?

### Blog paragraph

The outer product gave us a 3×3 matrix with nine entries. We now look for smaller families of entries that have their own consistent symmetry under rotation. We will first call them **pieces**; we will reserve the formal word *irreducible* until after we have counted and tested them.

The first piece is the symmetric part of `T`, defined by

$$
T_{\mathrm{sym}}=\frac{T+T^\top}{2}.
$$

In [ ]:
T_sym = (T + T.T)/2
T_sym

Because `T_sym.T = T_sym`, its entries mirror across the main diagonal: the upper- and lower-triangular parts match. It has three independent diagonal entries and three independent off-diagonal entries.

The second piece is the antisymmetric part:

$$
T_{\mathrm{anti}}=\frac{T-T^\top}{2}.
$$

In [ ]:
T_anti = (T - T.T)/2

T_anti

For an antisymmetric matrix, `T_anti.T = -T_anti`. Therefore its diagonal is zero, and each lower-triangular entry is the negative of its matching upper-triangular entry. Only three entries are independent.

We now remove the trace from the symmetric part to obtain the symmetric-traceless piece.

In [ ]:
tr_T = torch.trace(T)
T_ST = T_sym - tr_T/3 * torch.eye(3)
T_ST

The remaining trace piece is the isotropic part of the symmetric matrix. It has one scalar degree of freedom and is represented by

$$
T_{\mathrm{tr}}=\frac{\operatorname{tr}(T)}{3}I.
$$

The factor of `1/3` distributes the total trace equally across the three diagonal entries.

In [ ]:
T_tr = tr_T/3 * torch.eye(3)
T_tr

We now have all three pieces of the original tensor: the trace, antisymmetric, and symmetric-traceless parts. They reconstruct the original matrix:

$$
T=T_{\mathrm{tr}}+T_{\mathrm{anti}}+T_{\mathrm{ST}}.
$$

The following code checks reconstruction, zero trace, and the symmetry properties.

In [ ]:
T_reconstructed = T_tr + T_anti + T_ST
T_reconstructed

print(torch.allclose(T_reconstructed, T))
print(torch.trace(T_ST))
print(torch.allclose(T_ST, T_ST.T))

### Counting the independent numbers

We can now identify exactly which numbers are free to choose.

The trace piece contains one number, `t`:

$$
T_{\mathrm{tr}}=\begin{bmatrix}t&0&0\\0&t&0\\0&0&t\end{bmatrix}.
$$

The antisymmetric piece contains three independent upper-triangular values, `a`, `b`, and `c`; the other entries are forced:

$$
T_{\mathrm{anti}}=\begin{bmatrix}0&a&b\\-a&0&c\\-b&-c&0\end{bmatrix}.
$$

The symmetric-traceless piece contains three independent off-diagonal values and two independent diagonal values:

$$
T_{\mathrm{ST}}=\begin{bmatrix}p&q&r\\q&s&u\\r&u&-p-s\end{bmatrix}.
$$

The final diagonal entry is forced because the diagonal entries must sum to zero. Therefore the count is

$$
9=\underbrace{1}_{\text{trace}}+\underbrace{3}_{\text{antisymmetric}}+\underbrace{5}_{\text{symmetric-traceless}}.
$$

At this point, the count only shows that the three pieces retain all nine degrees of freedom. It does not yet prove that each piece remains closed under rotation; that is the next experiment.

Counting the pieces is not enough: we must check how they behave under rotation. We will compare two routes. In route A, rotate `T` and then decompose the rotated tensor. In route B, decompose `T` first and rotate each piece separately using $RXR^\top$. A matching total sum checks reconstruction, but the key test is that each corresponding pair of pieces matches individually. If it does, rotation does not cause information to leak from one family into another.

In [ ]:
T_rot = R @ T @ R.T
T_rot

Let us rotate T_tr first \(T_{\mathrm{tr}}' \stackrel{?}{=} R\,T_{\mathrm{tr}}\,R^\top.\)

In [ ]:
T_tr_rot = R @ T_tr @ R.T
T_ST_rot = R @ T_ST @ R.T
T_anti_rot = R @ T_anti @ R.T
print(T_tr_rot)
print(T_anti_rot)
print(T_ST_rot)


In [ ]:
print(T_tr_rot + T_ST_rot + T_anti_rot)
print(T_rot)

### Decompose the rotated tensor

Apply exactly the same trace, symmetric, antisymmetric, and symmetric-traceless formulas to `T_rot`. Do not introduce a new decomposition rule merely because the tensor has been rotated.

In [ ]:
T_rot_sym = (T_rot + T_rot.T) / 2
tr_T_rot = torch.trace(T_rot)
T_rot_tr = tr_T_rot / 3 * torch.eye(3)
T_rot_ST = T_rot_sym - T_rot_tr
T_rot_anti = (T_rot - T_rot.T) / 2

print("trace closed:", torch.allclose(T_rot_tr, T_tr_rot))
print("antisymmetric closed:", torch.allclose(T_rot_anti, T_anti_rot))
print("symmetric-traceless closed:", torch.allclose(T_rot_ST, T_ST_rot))

If all three checks print `True`, each piece follows the same rotation law on both routes. The trace, antisymmetric, and symmetric-traceless families are therefore closed under rotations: a rotation can mix components *within* one family, but cannot move information into a different family. This is the concrete observation that motivates the irreducible-representation language in the next section.

In [ ]:
# LAB 5 — discover 9 = 1 + 3 + 5: completed contracts.
assert torch.allclose(T_tr + T_anti + T_ST, T)
assert torch.allclose(T_anti.T, -T_anti)
assert torch.allclose(T_ST.T, T_ST)
assert torch.allclose(torch.trace(T_ST), torch.tensor(0.0), atol=1e-6)
assert 1 + 3 + 5 == T.numel()
assert torch.allclose(T_rot_tr, T_tr_rot)
assert torch.allclose(T_rot_anti, T_anti_rot)
assert torch.allclose(T_rot_ST, T_ST_rot)
print("9 = 1 + 3 + 5 and every piece is closed under rotation.")

## 6. Representations and irreducible representations

The experiment above motivates the formal language. A **representation** is a rule, written `D(R)`, that tells us how a feature's components change when the physical system is rotated by `R`. If a feature space separates into smaller subspaces that never mix with one another under rotation, those subspaces are independent rotational blocks. An **irreducible representation**, or **irrep**, is a block that cannot be split any further while preserving that property.

For three-dimensional rotations, irreps are labelled by a non-negative integer `l`. This `l` is a label for the rotational type; it is not the matrix rank and it is not a component index. A block of order `l` always contains `2l + 1` components.

The symbol `m` labels a component *inside one fixed-`l` block*. It runs through

$$
m=-l,-l+1,\ldots,0,\ldots,l-1,l.
$$

Therefore the three blocks discovered above are named

| Cartesian piece | Irrep label | Number of components | Component labels |
|---|---:|---:|---|
| trace | $l=0$ | $1$ | $m=0$ |
| antisymmetric / vector-like | $l=1$ | $3$ | $m=-1,0,+1$ |
| symmetric-traceless | $l=2$ | $5$ | $m=-2,-1,0,+1,+2$ |

The free Cartesian entries we used earlier are not yet the `m`-components. The `m` labels refer to coordinates in a spherical basis. A later change of basis will connect the Cartesian entries to those spherical components. Under rotation, the `m`-components within one `l` block generally mix with one another, but they do not mix with a different `l` block.

### Your explanation

Now attach formal names to the experiment. Explain what a representation `D(R)` acts on, why the same physical rotation uses a 1×1, 3×3, or 5×5 matrix on the three blocks above, and why `D(R₂R₁) = D(R₂)D(R₁)` is the feature-space version of your earlier rotation-composition check.

### ML bridge

Relate block structure to keeping different feature types in separate tensor blocks. A representation is the rule for how a feature vector changes; an irrep is a block that cannot be split into smaller rotation-closed blocks.

### Blog paragraph

The next cell is your original first draft. Preserve it, but revisit it only after completing Labs 4–6; annotate what changed in your understanding rather than silently replacing the draft.

> [!NOTE] Preserved early draft — not authoritative
> This paragraph records the learner's starting point. It uses incorrect terminology (including “Wagner” instead of **Wigner**) and should not be read as the explanation. The corrected account is in the surrounding section and Section 9.
>
> The D matrix is the Wagner D matrix for different orders of irreducible order. These are tensors of different size, so they require different rotational tensors. For example, for L = 0, that's a scalar rotation, which makes no difference at all, so it's just a scalar, 1, scaling by 1 for a vector. You usually have a 2-by-2 matrix or, depending on the number of dimensions, a 3-by-3 matrix. For higher-order tensors, you have higher-order matrices. You have these angular breakdowns, etc.

### Blog-ready explanation

A rotation does not act on every feature in the same way. A scalar, such as a temperature or an atom’s identity embedding, is unchanged, so its rotation rule is the one-by-one matrix $D^0(R)=[1]$. A three-dimensional vector does change, and its three components transform with the ordinary rotation matrix $D^1(R)=R$. The matrix $D^l(R)$ is therefore not one universal matrix: its size depends on the rotational type of the feature it acts on.

The nine entries of a $3\times3$ tensor give a useful example. They can be reorganised into three independent rotational blocks: a one-component trace ($l=0$), a three-component antisymmetric or vector-like part ($l=1$), and a five-component symmetric-traceless part ($l=2$). Each block has its own transformation matrix: $1\times1$, $3\times3$, or $5\times5$. The blocks may mix components internally, but a rotation cannot send information from one block into another.

This is what the word *irreducible* means here: the block is a smallest unit that remains closed under every rotation. The label $l$ identifies the block, while $m=-l,\ldots,l$ labels the individual components inside it. Thus $l$ tells us *which kind of rotational object* we have, and $m$ tells us *which coordinate within that object* we are looking at.

Finally, the representation matrices obey $D(R_2R_1)=D(R_2)D(R_1)$. This is simply the familiar rule that two physical rotations applied in sequence can be represented by multiplying their feature-space matrices in the same order. The MACE architecture uses these blocks so that neural-network operations can learn from features while preserving their precise behaviour under rotation.

In [ ]:
R1, _ = rotate_z_axis(torch.eye(3), math.pi/6)
R2, _ = rotate_z_axis(torch.eye(3), math.pi/12)

In [ ]:
print(R1)
print(R2)

In [ ]:
def D0(R):
    return torch.ones((1, 1), dtype=R.dtype, device=R.device)
def D1(R):
    return R

D0_R1 = D0(R1)
D0_R2 = D0(R2)
D1_R1 = D1(R1)
D1_R2 = D1(R2)


print(torch.allclose( (D1_R2 @ D1_R1), D1(R2 @ R1)) )
print(torch.allclose( (D0_R2 @ D0_R1), D0(R2 @ R1)) )

In [ ]:
# LAB 6 — first representations. D² is constructed with e3nn in Section 9.
assert D0_R1.shape == (1, 1)
assert D1_R1.shape == (3, 3)
assert torch.allclose(D0_R2 @ D0_R1, D0(R2 @ R1))
assert torch.allclose(D1_R2 @ D1_R1, D1(R2 @ R1), atol=1e-6)
representation_block_shapes = {0: (1, 1), 1: (3, 3), 2: (5, 5)}
print("representation shapes implied by 2l+1:", representation_block_shapes)
print("D(R2 R1) = D(R2) D(R1) for the scalar and vector blocks: True")

## 7. Angular orders `l` and components `m` in detail

### Your explanation

Build on the definitions above without assuming quantum mechanics. Explain the component count `2l + 1`, the allowed `m` values, why components with different `m` mix under rotation while `l` remains fixed, and why `(l_ξ, m_ξ)` denotes a pair belonging to factor `ξ` rather than equal values.

Create a table for `l = 0, 1, 2, 3` containing component count, allowed `m`, and geometric intuition.

### Blog paragraph

Once we have identified the rotational blocks, we need a compact way to name them. The integer $l$ labels the irreducible representation, or angular order; it is not one of the components inside that representation. Each fixed-$l$ block contains $2l+1$ components, indexed by $m=-l,\ldots,+l$. Thus $l=0$ has one component, $l=1$ has three, $l=2$ has five, and $l=3$ has seven.

Under a rotation, the components with different $m$ values may mix with one another, but the value of $l$ remains fixed. An $l=2$ feature can therefore rotate within its five-dimensional space, but it cannot turn into an $l=1$ or $l=3$ feature. For now, we are focusing only on this rotational bookkeeping; feature channels will be introduced later.

In [ ]:
list(range(-2, 2 + 1))

In [ ]:
def valid_l_m(max_order=3):
    l = range(max_order + 1)
    m = {i:list(range(-i, i+1)) for i in l}
    return l,m
l,m = valid_l_m(3)
print(l)
print(m)

In [ ]:
for i in l:
    assert len(m[i]) == 2*i+1

In [ ]:
# LAB 7 — angular index structure and per-order layout.
angular_layout = {l_order: {"m": m[l_order], "width": 2 * l_order + 1} for l_order in l}
for l_order, layout in angular_layout.items():
    assert len(layout["m"]) == layout["width"]
    print(f"l={l_order}: m={layout['m']} -> feature block (..., {layout['width']})")
print("Each row is a separate block; there is no fake fixed-width global m axis.")

## 8. Spherical harmonics — Fourier-like modes on the sphere

### Your explanation

Begin by connecting this section to the tensor experiment: the $3\times3$ tensor was decomposed into $l=0,1,2$ rotational blocks, while spherical harmonics provide a convenient direction-dependent basis for features living in those same kinds of blocks. Explain spherical harmonics as fixed direction-dependent basis functions and as a Fourier-like basis for functions on the sphere. Explain their input/output, the visual role of `l` and `m`, why distance is absent, why all `m` values for one `l` form a rotating block, and which real-basis normalization you use.

Use the `l = 1` real harmonics as the first bridge: check numerically whether they are proportional to the unit-direction components `x`, `y`, and `z` under your chosen ordering and normalization. Do not memorize the ordering; document it.

### ML bridge

Compare them with a fixed feature map or positional encoding while explaining the extra rotational structure. Then compare `l` with Fourier frequency and `m` with the components inside one frequency block; record both the power and the limits of this analogy.

### Blog paragraph

### Blog-ready explanation

At first, spherical harmonics may look like a completely new topic. They are actually the bridge from our Cartesian tensor experiment to the representation used by MACE. The tensor $T$ showed that rotational information naturally separates into blocks with $l=0$, $l=1$, and $l=2$. Spherical harmonics give us a convenient spherical basis for direction-dependent features in those same rotational types. We are not decomposing the same numerical tensor again; we are changing from Cartesian coordinates to a basis whose rotation law is already organised by $l$ and $m$.

A spherical harmonic is a fixed function on the surface of a sphere. It takes a direction, written as the unit vector $\hat{r}=r/\lVert r\rVert$, and returns a number describing how strongly that direction matches one particular angular pattern. For a fixed angular order $l$, we evaluate all $2l+1$ components, indexed by $m$, and treat them as one rotational feature block.

The order $l$ controls the angular complexity of the pattern. The $l=0$ harmonic is constant over the sphere, so it carries no directional detail. The $l=1$ harmonics are the simplest directional patterns; in a suitable real basis they are proportional to the three familiar coordinates $x$, $y$, and $z$ of a unit vector. Higher orders describe progressively finer angular variation. The exact ordering and signs depend on the basis convention, so we will document the convention rather than rely on memorisation.

Spherical harmonics deliberately ignore distance. That is useful because a molecular edge has two different kinds of information: *how far apart* two atoms are and *which way* one atom lies from the other. MACE handles the first with radial basis functions and the second with spherical harmonics. Keeping these roles separate gives the model a clean radial-versus-angular representation.

The important symmetry fact is that rotating the input direction does not destroy the feature; it rotates the whole fixed-$l$ block in a predictable way. With our convention, this is written as $Y^l(R\hat{r})=D^l(R)Y^l(\hat{r})$. The spherical harmonics provide the direction-dependent features, while the Wigner-$D$ matrix tells us how those features mix when the molecule is rotated. In machine-learning language, they are a structured, symmetry-aware positional encoding on the sphere—not arbitrary coordinates that would have to be relearned for every orientation.

### The missing bridge: from a Cartesian tensor to an $l=2$ block

Before using spherical harmonics, let us connect them carefully to the tensor decomposition we have already built. A tensor is not merely an array of numbers; it is an array together with a rule for how those numbers change under a transformation. In MACE the transformations of interest are three-dimensional rotations, so both the Cartesian and spherical descriptions below are representations of the same group, $SO(3)$. They organise the coordinates differently, but they do not describe different physics.

A Cartesian rank-two tensor $T$ is a $3\times3$ matrix that transforms as

$$
T \longmapsto RTR^{\mathsf T}.
$$

Its nine entries can be separated into three pieces that rotations never mix with one another:

$$
T_{\mathrm{tr}}=\frac{\operatorname{tr}(T)}{3}I,
\qquad
T_{\mathrm{anti}}=\frac{T-T^{\mathsf T}}{2},
\qquad
T_{\mathrm{ST}}=\frac{T+T^{\mathsf T}}{2}-T_{\mathrm{tr}}.
$$

The trace piece needs one independent number, the antisymmetric piece needs three, and the symmetric-traceless piece needs five. Therefore

$$
9=1+3+5.
$$

These three closed rotation types are called $l=0$, $l=1$, and $l=2$. Here $l$ is **not** the rank of the original Cartesian tensor. It labels an irreducible rotation type. An $l$ block always contains $2l+1$ components, and $m$ indexes the components inside that block. A rotation may mix the different $m$ components belonging to one $l$, but it cannot mix components belonging to different $l$ values.

Now begin with a molecular edge direction

$$
\hat r=(x,y,z),\qquad x^2+y^2+z^2=1,
$$

and form its outer product with itself:

$$
T=\hat r\hat r^{\mathsf T}
=
\begin{bmatrix}
x^2 & xy & xz\\
xy & y^2 & yz\\
xz & yz & z^2
\end{bmatrix}.
$$

This special $T$ is already symmetric, so $T_{\mathrm{anti}}=0$. Its trace is the scalar number $1$, but its **trace matrix piece** is $T_{\mathrm{tr}}=I/3$. The remaining information is

$$
T_{\mathrm{ST}}=\hat r\hat r^{\mathsf T}-\frac{I}{3}
=
\begin{bmatrix}
x^2-\frac13 & xy & xz\\
xy & y^2-\frac13 & yz\\
xz & yz & z^2-\frac13
\end{bmatrix}.
$$

This is the $l=2$ part of the directional outer product. It is also commonly called the **quadrupole tensor** and denoted $Q$; in this section, $Q$ is only shorthand for the object we already know: $Q:=T_{\mathrm{ST}}$.

Although the matrix displays nine entries, symmetry and the zero-trace condition leave only five independent numbers. One simple real Cartesian coordinate list is

$$
q_{\mathrm{cart}}=
(T_{xx},T_{yy},T_{xy},T_{xz},T_{yz}),
$$

where all entries on the right refer to $T_{\mathrm{ST}}$. The missing diagonal entry is forced to be $T_{zz}=-(T_{xx}+T_{yy})$. Thus the five numbers $(a,b,d,e,f)$ reconstruct the entire matrix as

$$
T_{\mathrm{ST}}=
\begin{bmatrix}
a & d & e\\
d & b & f\\
e & f & -a-b
\end{bmatrix}.
$$

This explains the connection to spherical harmonics. The operation $T\mapsto(T_{\mathrm{tr}},T_{\mathrm{anti}},T_{\mathrm{ST}})$ discovers the irreducible rotation types already present inside a Cartesian matrix. In contrast, the map

$$
\hat r\longmapsto Y^{(l)}(\hat r)
$$

directly manufactures a directional feature of a chosen rotation type. Up to normalization and a fixed change of coordinates,

$$
Y^{(0)}(\hat r)\sim 1,
\qquad
Y^{(1)}(\hat r)\sim\hat r,
\qquad
Y^{(2)}(\hat r)\sim\operatorname{coordinates}\!\left(\hat r\hat r^{\mathsf T}-\frac{I}{3}\right).
$$

So the five entries returned by a real $l=2$ spherical-harmonics routine contain the same *kind* of rotational information as the five independent entries of $T_{\mathrm{ST}}$. They are not usually the same five raw numbers: e3nn uses a fixed orthogonal, normalized real spherical basis, related to our Cartesian list by a constant change-of-basis matrix.

#### Where the complex basis fits

MACE and e3nn use real spherical harmonics, so no complex arithmetic is required here. The complex basis becomes pedagogically useful later, before Wigner-$D$ matrices and Clebsch–Gordan products, because rotations about the $z$ axis act especially simply on combinations such as $x\pm iy$. It clarifies the conventional labels $m=-l,\ldots,l$ and the selection rule $m_1+m_2=M$. The real and complex bases have the same expressive power; the real basis is convenient for real molecular data and avoids complex-valued storage and arithmetic.


In [ ]:
# LAB — derive the five independent l=2 coordinates from one direction.
r_hat = torch.tensor([1 / math.sqrt(2), 1 / math.sqrt(2), 0.0])
T_direction = torch.outer(r_hat, r_hat)
T_trace = torch.trace(T_direction) / 3 * torch.eye(3)
T_anti = (T_direction - T_direction.T) / 2
T_st = (T_direction + T_direction.T) / 2 - T_trace

l2_cartesian = torch.stack(
    [T_st[0, 0], T_st[1, 1], T_st[0, 1], T_st[0, 2], T_st[1, 2]]
)
a, b, d, e, f = l2_cartesian
T_st_reconstructed = torch.stack(
    [
        torch.stack([a, d, e]),
        torch.stack([d, b, f]),
        torch.stack([e, f, -a - b]),
    ]
)

assert torch.allclose(T_direction, T_trace + T_anti + T_st)
assert torch.allclose(T_anti, torch.zeros_like(T_anti))
assert torch.allclose(torch.trace(T_st), torch.tensor(0.0), atol=1e-6)
assert torch.allclose(T_st, T_st_reconstructed)

print("T = r_hat r_hat^T:\n", T_direction)
print("\nT_trace:\n", T_trace)
print("\nT_anti:\n", T_anti)
print("\nT_ST:\n", T_st)
print("\nFive independent Cartesian l=2 coordinates:\n", l2_cartesian)


In [ ]:
# Reused from papers/14-RFM/src/geometry.py.
def sample_sphere(n: int, device: torch.device | str | None = None) -> torch.Tensor:
    """Return n approximately uniform points on the unit sphere."""
    if n <= 0:
        raise ValueError("n must be positive")
    z = 2 * torch.rand(n, 1, device=device) - 1
    r = torch.sqrt(1 - torch.square(z))
    theta = 2 * torch.pi * torch.rand(n, 1, device=device) - 1
    x = (r * torch.cos(theta)).reshape(-1, 1)
    y = (r * torch.sin(theta)).reshape(-1, 1)
    return torch.concat([x, y, z], dim=1)

In [ ]:
n = 10
x= sample_sphere(n)
print(x.shape)
print(torch.linalg.vector_norm(x, dim=1))


In [ ]:
sqrt_pi = torch.sqrt(torch.tensor(torch.pi))
Y_0_0 = (0.5 / sqrt_pi).repeat(n)

In [ ]:
Y_0_0

In [ ]:
assert Y_0_0.shape == (n,)
assert torch.allclose(Y_0_0, Y_0_0[0].expand_as(Y_0_0))

fig = go.Figure(
    go.Scatter3d(
        x=x[:, 0].tolist(),
        y=x[:, 1].tolist(),
        z=x[:, 2].tolist(),
        mode="markers",
        marker={
            "size": 6,
            "color": Y_0_0.tolist(),
            "colorscale": "Viridis",
            "showscale": True,
            "colorbar": {"title": "Y₀⁰"},
        },
    )
)
fig.update_layout(
    title="l=0, m=0: constant spherical harmonic",
    scene={
        "xaxis_title": "x",
        "yaxis_title": "y",
        "zaxis_title": "z",
        "aspectmode": "data",
    },
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

### Using a standard spherical-harmonics implementation

At this point, we stop hand-writing increasingly complicated polynomial formulas. The formulas are useful for seeing the pattern, but they are not the right way to build a reliable implementation. Instead, we use the `e3nn` library, which evaluates the real spherical harmonics for us in a PyTorch-friendly representation.

The input is a batch of directions with shape `(n, 3)`. For a chosen order `l`, the routine returns a tensor with shape `(n, 2l+1)`: one row for each direction and one column for each component in that rotational block. With `normalize=True`, the input directions are treated as points on the unit sphere. With `normalization="integral"`, the output uses the same unit-integral convention as our earlier value $Y_0^0=1/\sqrt{4\pi}$.

This is not abandoning the mathematics. The earlier formulas tell us what kind of object the routine must produce: an `l=0` scalar block, an `l=1` vector-like block, an `l=2` five-component block, and so on. The library simply performs the careful polynomial evaluation and normalization. We still check the shapes, inspect the plots, and record the library's component ordering before calling those components `m` values.

The useful visual is a Fourier-style circle rather than a cloud of 3D sample points. We freeze the polar angle at the equator and let the direction travel once around the circle. Plotting each component against the azimuth shows the idea directly: `l=0` is flat, while higher `l` blocks contain richer angular variation. The `2l+1` curves in one row are not separate unrelated features; together they are the components of one rotational block. This is a one-dimensional slice of the sphere, chosen because it makes the angular-frequency intuition visible. The accompanying sphere plot restores the missing latitude direction and shows the same components as colored patterns over the whole surface.

In [ ]:
x

In [ ]:

def generate_spherical_harmonics(l,
                                 coords,
                                 normalize=True,
                                 normalization='integral'
                                 ):
    return o3.spherical_harmonics(
        l,
        coords,
        normalize,
        normalization
    )

n=3

l = {}
for i in range(n+1):
    l[i] = generate_spherical_harmonics(i, x)

In [ ]:
print(l[0])
print(l[1])

### The same $l=2$ information in e3nn's real basis

For the integral-normalized convention used here, e3nn's five real $l=2$ components are proportional to the following quadratic patterns, in the library's output order:

$$
\left(
\sqrt{\frac{15}{4\pi}}xz,
\sqrt{\frac{15}{4\pi}}xy,
\sqrt{\frac{5}{16\pi}}(2y^2-x^2-z^2),
\sqrt{\frac{15}{4\pi}}yz,
\sqrt{\frac{15}{16\pi}}(z^2-x^2)
\right).
$$

These expressions are not five new physical quantities. They are five particular linear combinations of the entries of $T_{\mathrm{ST}}$. There is therefore no separate ‘e3nn $Q$’. The geometric object is still

$$
Q:=T_{\mathrm{ST}}=\hat r\hat r^{\mathsf T}-\frac{I}{3}.
$$

What changes is only the coordinate system used to record its five independent numbers. Let

$$
q_{\mathrm{cart}}=(Q_{xx},Q_{yy},Q_{xy},Q_{xz},Q_{yz})^{\mathsf T}.
$$

Because $Q$ is traceless, $Q_{zz}=-Q_{xx}-Q_{yy}$. Define

$$
\alpha=\sqrt{\frac{15}{4\pi}},
\qquad
\beta=\sqrt{\frac{5}{16\pi}},
\qquad
\gamma=\sqrt{\frac{15}{16\pi}}.
$$

Then e3nn's real $l=2$ coordinate vector is obtained by one fixed matrix multiplication:

$$
Y^{(2)}_{\mathrm{e3nn}}(\hat r)
=Aq_{\mathrm{cart}},
\qquad
A=
\begin{bmatrix}
0&0&0&\alpha&0\\
0&0&\alpha&0&0\\
0&3\beta&0&0&0\\
0&0&0&0&\alpha\\
-2\gamma&-\gamma&0&0&0
\end{bmatrix}.
$$

For example, the third e3nn component initially looks unrelated to $Q$:

$$
\beta(2y^2-x^2-z^2).
$$

Substitute $x^2=Q_{xx}+1/3$, $y^2=Q_{yy}+1/3$, and $z^2=Q_{zz}+1/3$. The constant terms cancel, leaving

$$
\beta(2Q_{yy}-Q_{xx}-Q_{zz})=3\beta Q_{yy}.
$$

The fifth component similarly becomes $\gamma(Q_{zz}-Q_{xx})=\gamma(-2Q_{xx}-Q_{yy})$. The other three components are scaled versions of $Q_{xz}$, $Q_{xy}$, and $Q_{yz}$. This is the missing Cartesian-to-e3nn bridge.

The normalization makes e3nn's spherical functions orthonormal on the sphere, while the chosen combinations make rotations act through the standard real $l=2$ representation. The matrix $A$ is constant and invertible. It is not an orthogonal matrix under the naive Euclidean metric on our five-number list because $(Q_{xx},Q_{yy},Q_{xy},Q_{xz},Q_{yz})$ was a convenient independent list, not an orthonormal Cartesian basis.

The two descriptions also have equivalent rotation laws:

$$
Q' = RQR^{\mathsf T}
\qquad\Longleftrightarrow\qquad
Y^{(2)\prime}_{\mathrm{e3nn}}=D^{(2)}(R)Y^{(2)}_{\mathrm{e3nn}}.
$$

This is why the sphere plots show lobes: each color pattern is simply the positive and negative regions of one of these quadratic coordinate functions.

> **Takeaway.** e3nn does not create a new $Q$; it expresses the same $l=2$ tensor in a normalized real spherical basis.

For $\hat r=(1/\sqrt2,1/\sqrt2,0)$, the Cartesian five-number list is $(1/6,1/6,1/2,0,0)$. In e3nn's basis, the same direction produces approximately $(0,0.5463,0.1577,0,-0.2731)$. The numbers differ because the coordinate bases differ; both five-component blocks transform as $l=2$.


In [ ]:
# CHECK — reproduce e3nn's l=2 output from its documented real-basis polynomials.
x_coord, y_coord, z_coord = r_hat
manual_e3nn_l2 = torch.stack(
    [
        math.sqrt(15 / (4 * math.pi)) * x_coord * z_coord,
        math.sqrt(15 / (4 * math.pi)) * x_coord * y_coord,
        math.sqrt(5 / (16 * math.pi))
        * (2 * y_coord**2 - x_coord**2 - z_coord**2),
        math.sqrt(15 / (4 * math.pi)) * y_coord * z_coord,
        math.sqrt(15 / (16 * math.pi)) * (z_coord**2 - x_coord**2),
    ]
)
library_e3nn_l2 = generate_spherical_harmonics(2, r_hat)

alpha = math.sqrt(15 / (4 * math.pi))
beta = math.sqrt(5 / (16 * math.pi))
gamma = math.sqrt(15 / (16 * math.pi))
cartesian_to_e3nn = torch.tensor(
    [
        [0, 0, 0, alpha, 0],
        [0, 0, alpha, 0, 0],
        [0, 3 * beta, 0, 0, 0],
        [0, 0, 0, 0, alpha],
        [-2 * gamma, -gamma, 0, 0, 0],
    ],
    dtype=r_hat.dtype,
)
mapped_e3nn_l2 = cartesian_to_e3nn @ l2_cartesian

assert torch.allclose(manual_e3nn_l2, library_e3nn_l2, atol=1e-6)
assert torch.allclose(mapped_e3nn_l2, library_e3nn_l2, atol=1e-6)
print("Cartesian five-number list:", l2_cartesian)
print("A @ Cartesian list:", mapped_e3nn_l2)
print("e3nn normalized real basis:", library_e3nn_l2)


In [ ]:
from plotly.subplots import make_subplots

def plot_harmonics_on_equator(max_l=3, samples=361):
    """Show a Fourier-like azimuthal sweep of each angular block."""
    phi = torch.linspace(0, 2 * math.pi, samples)
    directions = torch.stack(
        (torch.cos(phi), torch.sin(phi), torch.zeros_like(phi)),
        dim=1,
    )
    circle_blocks = {
        l_order: generate_spherical_harmonics(l_order, directions)
        for l_order in range(max_l + 1)
    }

    fig = make_subplots(
        rows=max_l + 1,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[
            f"l={l_order}: {2 * l_order + 1} components"
            for l_order in range(max_l + 1)
        ],
    )
    phi_degrees = torch.rad2deg(phi).tolist()
    for l_order, block in circle_blocks.items():
        for component in range(block.shape[1]):
            fig.add_trace(
                go.Scatter(
                    x=phi_degrees,
                    y=block[:, component].detach().cpu().tolist(),
                    mode="lines",
                    name=f"component {component}",
                    legendgroup=f"l={l_order}",
                    showlegend=l_order == 1,
                ),
                row=l_order + 1,
                col=1,
            )
    fig.update_xaxes(title_text="azimuth around the circle (degrees)", row=max_l + 1, col=1)
    fig.update_yaxes(title_text="basis value")
    fig.update_layout(
        title="Spherical harmonics as Fourier-like angular modes",
        height=230 * (max_l + 1),
        width=850,
        hovermode="x unified",
        margin={"l": 60, "r": 20, "b": 60, "t": 70},
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return circle_blocks

circle_blocks = plot_harmonics_on_equator(max_l=3)

def plot_harmonics_on_sphere(l_orders=(0, 1, 2), n_theta=31, n_phi=61):
    """Display spherical-harmonic values as colored sphere surfaces."""
    theta = torch.linspace(0, math.pi, n_theta)
    phi = torch.linspace(0, 2 * math.pi, n_phi)
    theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing="ij")
    sphere_xyz = torch.stack(
        (
            torch.sin(theta_grid) * torch.cos(phi_grid),
            torch.sin(theta_grid) * torch.sin(phi_grid),
            torch.cos(theta_grid),
        ),
        dim=-1,
    )
    flat_xyz = sphere_xyz.reshape(-1, 3)
    sphere_blocks = {
        l_order: generate_spherical_harmonics(l_order, flat_xyz).reshape(
            n_theta, n_phi, 2 * l_order + 1
        )
        for l_order in l_orders
    }

    panels = [(l_order, component)
              for l_order in l_orders
              for component in range(2 * l_order + 1)]
    columns = 2
    rows = math.ceil(len(panels) / columns)
    specs = [
        [
            {"type": "surface"} if row * columns + col < len(panels) else None
            for col in range(columns)
        ]
        for row in range(rows)
    ]
    fig = make_subplots(rows=rows, cols=columns, specs=specs,
                        subplot_titles=[f"l={l_order}, component {component}"
                                        for l_order, component in panels])

    for index, (l_order, component) in enumerate(panels):
        row, col = divmod(index, columns)
        values = sphere_blocks[l_order][:, :, component]
        limit = max(float(values.abs().max()), 1e-8)
        fig.add_trace(
            go.Surface(
                x=sphere_xyz[:, :, 0].tolist(),
                y=sphere_xyz[:, :, 1].tolist(),
                z=sphere_xyz[:, :, 2].tolist(),
                surfacecolor=values.tolist(),
                colorscale="RdBu",
                cmin=-limit,
                cmax=limit,
                showscale=index == 0,
                colorbar={"title": "value"},
                hovertemplate="%{surfacecolor:.3f}<extra></extra>",
            ),
            row=row + 1,
            col=col + 1,
        )
    fig.update_scenes(aspectmode="data")
    fig.update_layout(
        title="Spherical harmonics on the full sphere",
        height=360 * rows,
        width=850,
        showlegend=False,
        hoverlabel={"align": "left", "font": {"size": 14}},
        margin={"l": 0, "r": 0, "b": 0, "t": 70},
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return sphere_blocks

sphere_blocks = plot_harmonics_on_sphere()

In [ ]:
# LAB 8A — component counts and the concrete l=1 Cartesian bridge.
bridge_directions = sample_sphere(256)
Y1_bridge = generate_spherical_harmonics(1, bridge_directions)
integral_normalization_l1 = math.sqrt(3 / (4 * math.pi))

assert Y1_bridge.shape == (256, 3)
assert torch.allclose(
    Y1_bridge, integral_normalization_l1 * bridge_directions, atol=1e-6
)
for l_order, block in circle_blocks.items():
    assert block.shape[1] == 2 * l_order + 1
print("e3nn l=1 ordering is proportional to (x,y,z) under this convention.")
print("Every plotted l block has exactly 2l+1 component curves.")

In [ ]:
# LAB 8B — Monte Carlo orthonormality check for the integral-normalized basis.
torch.manual_seed(7)
gram_directions = sample_sphere(12000)
gram_errors = {}
for l_order in [0, 1, 2]:
    Y_l = generate_spherical_harmonics(l_order, gram_directions)
    estimated_gram = (4 * math.pi / gram_directions.shape[0]) * (Y_l.T @ Y_l)
    identity = torch.eye(2 * l_order + 1, dtype=estimated_gram.dtype)
    gram_errors[l_order] = torch.max(torch.abs(estimated_gram - identity)).item()
    assert gram_errors[l_order] < 0.04
print("Monte Carlo Gram max errors by l:", gram_errors)
print("Takeaway: components within each fixed-l basis are approximately orthonormal.")

## 9. Wigner-D matrices

### Learning prompt (preserved)

Use the plots above to motivate Wigner-D. Explain what rotates physically, what rotates in feature space, why `Dˡ(R)` has shape `(2l + 1) × (2l + 1)`, and what the identity `Yˡ(Rr̂) = Dˡ(R)Yˡ(r̂)` claims.

### The same physical rotation in feature space

Let $R$ rotate a physical edge direction $\hat{\mathbf r}$. There are two routes:

$$
\underbrace{Y^{(l)}(R\hat{\mathbf r})}_{\text{rotate direction, then encode}}
=
\underbrace{D^{(l)}(R)Y^{(l)}(\hat{\mathbf r})}_{\text{encode, then rotate feature}}.
$$

$Y^{(l)}$ returns one block with $2l+1$ components, so $D^{(l)}(R)$ has shape $(2l+1,2l+1)$. For $l=0$ it is $[1]$; for $l=1$ it acts on three vector-like components; for $l=2$ it mixes the five components of the symmetric-traceless type. Components inside one block may mix, but rotation never leaks information between different $l$ blocks.

> **Takeaway:** $R$ acts on physical coordinates; $D^{(l)}(R)$ describes that same rotation in the coordinates of an $l$-type feature.

In [ ]:
R

In [ ]:
def generate_wigner_D(l, R):
    # This notebook passes proper rotations only (det R = +1). e3nn requires
    # a parity argument, but parity does not affect D(R) on this SO(3) subset.
    return o3.Irrep(l, 1).D_from_matrix(R)

D = {}
for i in range(3):
    D[i] = generate_wigner_D(i, R)

print([D[i].shape for i in range(3)])



In [ ]:
# LAB 9A — inspect Wigner-D actions without adding a decorative plot.
for l_order in [0, 1, 2]:
    expected_width = 2 * l_order + 1
    assert D[l_order].shape == (expected_width, expected_width)
    assert torch.allclose(
        D[l_order].T @ D[l_order], torch.eye(expected_width), atol=1e-5
    )
print("D^l shapes:", {l_order: tuple(D[l_order].shape) for l_order in D})
print("Each D^l is orthogonal: it mixes components inside one block while preserving its norm.")

In [ ]:
Y2_r_hat = generate_spherical_harmonics(2, r_hat)
r_hat_rot = R @ r_hat
Y2_r_hat_rot = generate_spherical_harmonics(2, r_hat_rot)
print(r_hat, Y2_r_hat)
print(r_hat_rot, Y2_r_hat_rot)
Y2_pred = D[2] @ Y2_r_hat
print(Y2_pred)
torch.max(torch.abs(Y2_pred - Y2_r_hat_rot))

In [ ]:
# LAB 9B — numerical equivariance and representation composition.
wigner_test_directions = sample_sphere(128)
wigner_tolerance = 1e-5
wigner_errors = {}
composition_errors = {}
R1_for_composition = R1.to(torch.float64)
R2_for_composition = R2.to(torch.float64)
for l_order in [0, 1, 2]:
    D_R = generate_wigner_D(l_order, R)
    Y_before = generate_spherical_harmonics(l_order, wigner_test_directions)
    Y_after_input_rotation = generate_spherical_harmonics(
        l_order, wigner_test_directions @ R.T
    )
    Y_after_feature_rotation = Y_before @ D_R.T
    wigner_errors[l_order] = torch.max(
        torch.abs(Y_after_input_rotation - Y_after_feature_rotation)
    ).item()
    # Promote this sensitive composition check: e3nn's float32 angle extraction
    # can accumulate about 3e-5 error even when the representation law is correct.
    D_composed = generate_wigner_D(
        l_order, R2_for_composition @ R1_for_composition
    )
    D_sequential = generate_wigner_D(
        l_order, R2_for_composition
    ) @ generate_wigner_D(l_order, R1_for_composition)
    composition_errors[l_order] = torch.max(torch.abs(D_composed - D_sequential)).item()
    assert wigner_errors[l_order] < wigner_tolerance
    assert composition_errors[l_order] < wigner_tolerance
print("Y(Rr) versus D(R)Y(r) max errors:", wigner_errors)
print("D(R2 R1) versus D(R2)D(R1) max errors:", composition_errors)

## 10. Feature channels versus angular components

### Your explanation

Distinguish channel `k`, angular order `l`, and component `m`. Explain `W[k, k̃, l]` as a learned channel map whose action is shared across every `m` component of one `l` block.

### ML bridge

Connect this to an ordinary dense layer with a symmetry-required tied axis.

### Blog paragraph

### Blog-ready explanation

It is easy to confuse a feature channel with an angular component because both appear as axes of one tensor. They do different jobs. The index $l$ selects the rotation type: $l=0$ is scalar-like, $l=1$ is vector-like, and $l=2$ is the five-component symmetric-traceless type. The index $m$ then identifies one of the $2l+1$ components *inside* that chosen rotation type. A channel index, written $k$, is different again: it labels separate learned copies of the same kind of geometric feature.

For example, suppose we hold $l=2$ fixed. Two input channels then form a feature array $H_{\tilde k m}$ of shape $(2,5)$: two learned copies of a five-component $l=2$ object. A learned channel map $W_{k\tilde k}$ of shape $(3,2)$ creates three output channels:

$$
H'_{km}=\sum_{\tilde k}W_{k\tilde k}H_{\tilde k m}.
$$

The result has shape $(3,5)$. The matrix $W$ decides how to combine the two input channels into three output channels, but it applies exactly the same channel mixture at every $m$. In ordinary deep-learning language, this is a dense layer across channels with weights tied across the angular-component axis.

That tying is not a cosmetic implementation choice. Under rotation, the five $m$ components mix through the same Wigner-$D^{(2)}$ matrix. Because $W$ never singles out a particular $m$, channel mixing commutes with that rotation. If we gave each $m$ its own independent weight matrix, the channel map would generally fail to commute with Wigner-$D$, and the layer would no longer be equivariant.


In [ ]:
Y2_r_hat_rot

In [ ]:
mix = torch.nn.Linear(2,3,False)
Hin = torch.concat([Y2_r_hat.reshape(1,-1), Y2_r_hat_rot.reshape(1,-1)], dim=0)
Hin.shape
Hout = mix(Hin.T).T
Hout.shape

print(mix(D[2] @ Hin.T))
# print(D[2] @ mix(Hin.T).T)


In [ ]:
# LAB 10 — channel mixing: valid shared weights versus invalid m-dependent weights.
D2 = D[2].to(dtype=Hin.dtype)

# Each row is an l=2 feature block, so rotation acts on the right.
Hin_rotated = Hin @ D2.T
Hout_from_rotated_input = mix(Hin_rotated.T).T
Hout_then_rotated = Hout @ D2.T
shared_weight_error = torch.max(
    torch.abs(Hout_from_rotated_input - Hout_then_rotated)
)

# Give every m component its own channel map: this should break equivariance.
torch.manual_seed(0)
m_dependent_weights = torch.randn(5, 3, 2, dtype=Hin.dtype)

def m_dependent_mix(features):
    return torch.einsum("moi,im->om", m_dependent_weights, features)

Hout_bad = m_dependent_mix(Hin)
Hout_bad_from_rotated_input = m_dependent_mix(Hin_rotated)
Hout_bad_then_rotated = Hout_bad @ D2.T
m_dependent_error = torch.max(
    torch.abs(Hout_bad_from_rotated_input - Hout_bad_then_rotated)
)

assert torch.allclose(
    Hout_from_rotated_input, Hout_then_rotated, atol=1e-5
)
assert m_dependent_error > 1e-4

print(f"{'channel map':<22} | max equivariance error")
print(f"{'shared across m':<22} | {shared_weight_error:.2e}")
print(f"{'m-dependent':<22} | {m_dependent_error:.2e}")

## 11. Why products create new rotation types

### The question that remains

So far, we have learned two complementary lessons. Spherical harmonics manufacture features of a chosen rotation type $l$, and channel mixing can combine learned channels *within* a fixed $l$ block. That is deliberately restrictive: an equivariant linear map cannot freely turn an $l=1$ vector-like feature into an $l=2$ feature. To create richer geometric information, we need a new operation: multiply two equivariant features.

Take the simplest case: two $l=1$ vectors, $v$ and $w$. Their pairwise products form the outer product

$$
T=v\otimes w,\qquad T_{ij}=v_iw_j.
$$

This gives nine values. We already proved in Section 4 that they transform equivariantly as $T\mapsto RTR^{\mathsf T}$. The new point is how to *read* that same fact: $T$ is one feature living in the nine-dimensional tensor-product representation

$$
l=1\otimes l=1.
$$

In flattened coordinates, a rotation acts by $D^{(1)}(R)\otimes D^{(1)}(R)$ on those nine values. This is the representation-theory version of the familiar Cartesian rule $RTR^{\mathsf T}$.

### Why the product is not finished yet

The nine-dimensional product is equivariant, but it is not irreducible: it contains several distinct rotation types mixed together. This is not a new defect in the outer product. It is exactly the structure we discovered earlier when we split a $3\times3$ tensor into trace, antisymmetric, and symmetric-traceless pieces.

Section 12 reinterprets that old decomposition as the answer to a new question: given two input irreps, which irreps can their product contain, and how do we extract each one? Clebsch–Gordan coefficients are the general rule for performing that extraction.

### Blog-ready takeaway

Linear channel mixing preserves a feature's rotation type. Tensor products are the mechanism that create richer types: multiplying two $l=1$ vector blocks gives a nine-component equivariant product space. That space is still reducible, so the next step is to reorganise it into its independent $l=0$, $l=1$, and $l=2$ parts. This is the job of Clebsch–Gordan decomposition.

### Checkpoint — no new code

Section 4 already established the only numerical fact needed here:

$$
(Rv)\otimes(Rw)=R(v\otimes w)R^{\mathsf T}.
$$

Keep that code as the Cartesian proof. The sole new annotation is: its input is $l=1\otimes l=1$, and its nine-value output is an equivariant but reducible product representation. The actual decomposition begins next.

## 12. Clebsch–Gordan decomposition

### Start from the decomposition we already know

Section 4 already decomposed the Cartesian outer product $T=v\otimes w$ into trace, antisymmetric, and symmetric-traceless pieces. Do **not** redo that work here. This section gives that same decomposition its representation-theory meaning: it is the simplest Clebsch–Gordan (CG) change of basis.

For two ordinary vectors, the decomposition is:

| Piece of $T=v\otimes w$ | Output rotation type | Independent components |
| --- | --- | ---: |
| Trace part $T_{\mathrm{tr}}$ | $l=0$ | 1 |
| Antisymmetric part $T_{\mathrm{anti}}$ | $l=1$ | 3 |
| Symmetric-traceless part $T_{\mathrm{ST}}$ | $l=2$ | 5 |

$$
1\otimes1=0\oplus1\oplus2,
\qquad
9=1+3+5.
$$

No new tensor is being created here: Clebsch–Gordan decomposition simply reorganises the same nine outer-product values into three blocks that rotate independently.

### Blog paragraph

#### From a pile of products to meaningful geometric features

An outer product is the most literal way to let two geometric features interact. If $v$ and $w$ are vector features, their product $T=v\otimes w$ contains every pairwise coordinate product: $v_xw_x$, $v_xw_y$, and so on. There are $3\times3=9$ such values. The product transforms correctly when we rotate the molecule, but its nine numbers are initially just a mixed pile of different rotation behaviours.

The useful move is to reorganise that pile into blocks that do not mix under rotation. For two vectors, the trace of $T$ is one scalar number, the antisymmetric part behaves like a three-component vector, and the symmetric-traceless part is a five-component quadrupole. In the language of irreducible representations, this is

$$
1\otimes1=0\oplus1\oplus2.
$$

The dimensions provide a reassuring accounting check: the original $9$ values have not disappeared or been approximated; they have been regrouped as $9=1+3+5$. This is why a tensor product can make a network more expressive without breaking rotational equivariance: it creates richer geometric information, then keeps each distinct rotation type in its own block.

This pattern is completely general. Combining an $l_1$ block with an $l_2$ block can produce every total angular order from $|l_1-l_2|$ through $l_1+l_2$:

$$
l_1\otimes l_2=\bigoplus_{L=|l_1-l_2|}^{l_1+l_2}L.
$$

For example, a vector block has three components and an $l=2$ block has five, so their $15$ raw pairwise products reorganise as

$$
1\otimes2=1\oplus2\oplus3,
\qquad
3\times5=3+5+7.
$$

Clebsch–Gordan coefficients are the fixed, symmetry-dictated recipe for this regrouping. They take pairs of input components labelled $(m_1,m_2)$ and form output components labelled $(L,M)$. They are not learned parameters: geometry determines them once we choose a basis convention. A MACE layer learns which *channels* to combine; Clebsch–Gordan coefficients determine which combinations have a clean, well-defined rotation type.

The component label is conserved: $M=m_1+m_2$. But $M$ does not uniquely determine the output rotation type. For example, $M=1$ may live in an $L=1$, $L=2$, or $L=3$ block; the Clebsch–Gordan coefficients separate those distinct rotation behaviours.

> **Curious reader?** The standard derivation treats CG coefficients as the change of basis between the raw product basis and total-angular-order basis. [These notes on angular-momentum addition](https://etneil.github.io/grad_qm_lec_notes/addition_J.html) give the full route; it is optional for following MACE.

The lab below makes this fixed change of basis concrete for $1\otimes1$. Later, when we use the complex spherical basis, the same idea is written with the component labels $(m_1,m_2)$ and $(L,M)$; there the exact selection rule is $M=m_1+m_2$.

In [ ]:
print(torch.trace(torch.outer(v,w)))

The trace is the unnormalised $l=0$ coordinate: rotating both input vectors leaves $\operatorname{tr}(v\otimes w)=v\cdot w$ unchanged. The normalised scalar used in the lab is $s=\operatorname{tr}(T)/\sqrt3$.

### Lab convention: one explicit real CG-style matrix

This lab uses a real Cartesian basis, not e3nn's real spherical-harmonic basis. That is deliberate: the geometry is visible here. Different valid basis conventions may change signs or rotate coordinates *within* one $l$-block, but never mix the $l=0$, $l=1$, and $l=2$ blocks.

Flatten the raw product row-by-row:

$$
t=(T_{xx},T_{xy},T_{xz},T_{yx},T_{yy},T_{yz},T_{zx},T_{zy},T_{zz})^\mathsf{T}.
$$

We will create a normalized output vector $y=C_{\mathrm{cart}}t$ in this order:

$$
y=(s,a_x,a_y,a_z,q_1,q_2,q_3,q_4,q_5)^\mathsf{T},
$$

where

$$
\begin{aligned}
s &= \frac{T_{xx}+T_{yy}+T_{zz}}{\sqrt3},\\
(a_x,a_y,a_z) &= \frac{1}{\sqrt2}(T_{yz}-T_{zy},\;T_{zx}-T_{xz},\;T_{xy}-T_{yx}),\\
(q_1,q_2,q_3,q_4,q_5) &= \left(\frac{T_{xx}-T_{yy}}{\sqrt2},\;\frac{2T_{zz}-T_{xx}-T_{yy}}{\sqrt6},\;\frac{T_{xy}+T_{yx}}{\sqrt2},\;\frac{T_{xz}+T_{zx}}{\sqrt2},\;\frac{T_{yz}+T_{zy}}{\sqrt2}\right).
\end{aligned}
$$

The first coordinate is the normalized $l=0$ scalar; the next three are the $l=1$ block; the final five are the $l=2$ block. These equations are the rows of $C_{\mathrm{cart}}$. The $1/\sqrt3$ normalization is why reconstructing the trace *matrix* still gives $T_{\mathrm{tr}}=\operatorname{tr}(T)I/3$.

In [ ]:
# LAB 12A — CG as a real change of basis for 1 ⊗ 1
# You already decomposed T in Section 4. Do not repeat that calculation.
#
# 1. Form T = torch.outer(v, w), then flatten row-by-row:
#    t = (Txx, Txy, Txz, Tyx, Tyy, Tyz, Tzx, Tzy, Tzz).
# 2. Build the fixed 9×9 matrix C_cart from the convention in the Markdown cell above.
# 3. Compute y = C_cart @ t. Its slices are l=0 (one value), l=1 (three), l=2 (five).
#
# CHECK: C_cart.shape == (9, 9) and y.shape == (9,).
# CHECK: C_cart.T @ C_cart is the 9×9 identity.
# CHECK: C_cart.T @ y reconstructs t.

In [ ]:
T = torch.outer(v,w)
t = T.flatten()
sqrt_3_inv = torch.tensor(3**(-0.5))
c0 = torch.tensor([sqrt_3_inv, 0, 0, 0, sqrt_3_inv, 0, 0, 0, sqrt_3_inv])
c0

s = c0 @ t
print(s)
print(torch.trace(T) / math.sqrt(3))


In [ ]:
sqrt_2_inv = 2**(-0.5)
cax = torch.tensor([0, 0, 0, 0, 0, sqrt_2_inv, 0, -sqrt_2_inv, 0])
cax
cay = torch.tensor([0, 0, -sqrt_2_inv, 0, 0, 0, sqrt_2_inv, 0, 0])
caz = torch.tensor([0, sqrt_2_inv,0,-sqrt_2_inv, 0, 0, 0, 0, 0])

print(cax.shape, cay.shape, caz.shape)
cl1 = torch.concat([cax.reshape(1,-1), cay.reshape(1,-1), caz.reshape(1,-1)], dim=0)
print(cl1.shape)
print(cl1 @ cl1.T)

In [ ]:
sqrt_6_inv = sqrt_3_inv * sqrt_2_inv
cq1 = torch.tensor([sqrt_2_inv, 0, 0, 0, -sqrt_2_inv, 0, 0, 0, 0])
cq2 = torch.tensor([-sqrt_6_inv, 0, 0, 0, -sqrt_6_inv, 0, 0, 0, 2 * sqrt_6_inv])
cq3 = torch.tensor([0, sqrt_2_inv, 0, sqrt_2_inv, 0, 0, 0, 0, 0])
cq4 = torch.tensor([0, 0, sqrt_2_inv, 0, 0, 0, sqrt_2_inv, 0, 0])
cq5 = torch.tensor([0, 0, 0, 0, 0, sqrt_2_inv, 0, sqrt_2_inv, 0])

C_cart = torch.stack([c0, cax, cay, caz, cq1, cq2, cq3, cq4, cq5])
y = C_cart @ t

assert torch.allclose(C_cart.T @ C_cart, torch.eye(9))
assert torch.allclose(C_cart.T @ y, t)
print(C_cart.shape, y.shape)
print("C_cart.T @ C_cart = I and C_cart.T @ y = t ✓")

In [ ]:
# LAB 12B — read the three output blocks
l0_block, l1_block, l2_block = y[:1], y[1:4], y[4:9]

assert torch.allclose(l0_block[0], torch.trace(T) / math.sqrt(3))
print("l=0 block:", l0_block.shape)
print("l=1 block:", l1_block.shape)
print("l=2 block:", l2_block.shape)
print("The same nine raw products are now organised as 1 + 3 + 5.")

# C_cart is fixed by the rotation symmetry and our chosen normalized basis; it is not learned.
# STOP HERE: complex-basis coefficient tables and the M = m₁ + m₂ selection rule are a later optional deep dive.

In [ ]:
(C_cart.T @ C_cart).shape

In [ ]:
print(torch.allclose(C_cart.T @ C_cart, torch.eye(9)))
print(torch.allclose(C_cart.T @ y, t))

## 13. Higher-order coupling: `ν` and `η_ν`

### Your explanation

Using three factors, explain what `ν` counts, how tensor-product order relates to physical body order, why intermediate angular orders appear, and why `η_ν` is needed to distinguish multiple valid coupling paths to the same final `L`. Draw at least two coupling trees.

### Blog paragraph

#### From one product to higher-order local geometry

Section 12 coupled two feature blocks once. MACE repeats that same operation. Each repetition lets an atom's representation depend on a richer joint pattern in its local neighbourhood: not merely one neighbour at a time, but combinations of several neighbour-derived features. The important point is that every intermediate result is still sorted into clean rotation types.

The symbol $\nu$ counts how many such factors are coupled together in one higher-order feature. In an atom-centred message, $\nu$ neighbour-derived factors can create dependence on up to $\nu+1$ atoms once the centre atom is included. Thus increasing $\nu$ is MACE's controlled way to increase local body order without explicitly looping over every neighbour triplet, quadruplet, and so on.

There is one small complication: with three or more factors, the same final rotation type can be reached through more than one valid intermediate route. Take three vector blocks as the smallest example:

$$
(1\otimes1)\otimes1.
$$

The first pair may couple to an intermediate order $0$, $1$, or $2$. If the desired final output has $L=1$, all three routes are valid:

$$
(1\otimes1\to0)\otimes1\to1,\qquad
(1\otimes1\to1)\otimes1\to1,\qquad
(1\otimes1\to2)\otimes1\to1.
$$

$\eta_\nu$ is simply a label for this coupling route. It is not a learned feature and it is not another spatial coordinate. It tells MACE which precomputed generalised Clebsch–Gordan map was used to turn the raw product into a particular $(L,M)$ output block. Keeping this label explicit prevents different valid routes to the same $L$ from being accidentally treated as the same geometric object.

This is the pattern to keep in mind when Equation 10 arrives: MACE first mixes ordinary feature channels, then multiplies selected factors, then uses a fixed symmetry rule—identified by $\eta_\nu$—to regroup the result into a well-defined output rotation type.

In [ ]:
l12 = [0, 1, 2]
l3 = 1
for i in l12:
    print(list(range(abs(i-l3), i+l3+1)))



          



In [ ]:
# LAB 13 — enumerate nu=3 paths for 1 x 1 x 1 -> L=1.
factor_orders_nu3 = (1, 1, 1)
target_L_nu3 = 1
valid_intermediate_orders = list(range(abs(1 - 1), 1 + 1 + 1))
eta3_paths = {
    f"p_{l12}": {"l12": l12, "route": ((1, 1, l12), (l12, 1, target_L_nu3))}
    for l12 in valid_intermediate_orders
    if abs(l12 - 1) <= target_L_nu3 <= l12 + 1
}
assert set(path["l12"] for path in eta3_paths.values()) == {0, 1, 2}
assert all(abs(a - b) <= out <= a + b for path in eta3_paths.values() for a, b, out in path["route"])
print("nu=3 path labels and intermediate orders:", eta3_paths)
print("Each path ends in an L=1 block with 2L+1 = 3 components.")

## 14. Radial information

### Learning prompt (preserved)

Explain distance versus direction, fixed Bessel basis functions, the smooth cutoff, learned combinations of fixed basis values, and why radial outputs are rotationally invariant.

### ML bridge

Compare fixed radial expansion followed by learned mixing with a fixed feature map feeding an MLP.

#### Direction tells us the shape; distance tells us the scale

The angular machinery so far describes how a neighbour sits *around* a central atom. Spherical harmonics distinguish directions, but they deliberately ignore whether that neighbour is one ångström away or four. A molecular interaction needs both pieces: direction tells the network the geometry of an arrangement; distance tells it the scale at which that arrangement occurs.

For mode $n=1,\ldots,P$ and cutoff radius $r_{\mathrm{cut}}$, the function implemented below is

$$
b_n(r)=\sqrt{\frac{2}{r_{\mathrm{cut}}}}\,\frac{\sin(n\pi r/r_{\mathrm{cut}})}{r},\qquad
b_n(0)=\sqrt{\frac{2}{r_{\mathrm{cut}}}}\,\frac{n\pi}{r_{\mathrm{cut}}}.
$$

`torch.sinc` evaluates this expression safely at $r=0$. For `E` edge distances and `P` modes, evaluating all modes gives a fixed feature matrix with shape `(E, P)`. Each column responds at a different radial frequency; the basis functions themselves are not learned.

The smooth envelope is

$$
f_{\mathrm{cut}}(r)=\begin{cases}\tfrac12[\cos(\pi r/r_{\mathrm{cut}})+1],&r<r_{\mathrm{cut}},\\0,&r\ge r_{\mathrm{cut}},\end{cases}
\qquad \widetilde R_n(r)=f_{\mathrm{cut}}(r)b_n(r).
$$

It brings every basis value and its first derivative to zero at the boundary. Atoms beyond the local neighbourhood therefore exchange no direct message, without an abrupt energy or force jump when an edge crosses the cutoff.

The learnable part comes *after* this fixed expansion. A linear layer or multilayer perceptron maps each edge row `(P,)` to the radial coefficients needed by a chosen MACE coupling. It is therefore not choosing one Bessel function and discarding the rest; it learns mixtures of the full radial feature vector.

Most importantly, distance is unchanged by a rotation. The radial branch is therefore an $l=0$ scalar input: it may modulate an equivariant angular feature without changing that feature's rotation type.

> **Takeaway:** the radial network learns *how much* an edge contributes; spherical harmonics determine *how that contribution rotates*.

In [ ]:
def bessel_basis(r, n, r_cut):
    """Evaluate mode n at distances r; the output has the same shape as r."""
    normalization = math.sqrt(2 / r_cut)
    frequency = n * math.pi / r_cut
    return normalization * torch.sinc(n * r / r_cut) * frequency


In [ ]:
r_cut = 5.
r = torch.linspace(0, r_cut, 400)


In [ ]:
# LAB 14 — radial basis and cutoff
import matplotlib.pyplot as plt

r_cut = 5.0
r = torch.linspace(0, r_cut, 400)

plt.figure(figsize=(8, 4))
for n in [1, 2, 3]:
    plt.plot(r, bessel_basis(r, n, r_cut), label=f"n = {n}")

plt.axvline(r_cut, color="black", linestyle="--", alpha=0.5)
plt.xlabel("Distance r")
plt.ylabel("Basis value")
plt.title("Question: what distance patterns do three fixed modes detect?")
plt.legend()
plt.grid(alpha=0.2)
plt.show()
print("Takeaway: one distance becomes several fixed measurements at different radial frequencies.")

# Reading the plot: larger n produces more zero crossings before r_cut.
# The learned linear combination is plotted in the completed cell below.
assert all(torch.isclose(bessel_basis(torch.tensor(r_cut), n, r_cut), torch.tensor(0.0), atol=1e-6) for n in [1, 2, 3])

In [ ]:
def cosine_cutoff(r, r_cut):
    """Smooth envelope: 1 at r=0, 0 (with zero slope) at r=r_cut."""
    inside = 0.5 * (torch.cos(math.pi * r / r_cut) + 1.0)
    return torch.where(r < r_cut, inside, torch.zeros_like(r))


In [ ]:
# One radial mode before and after applying the smooth cutoff envelope.
n = 1
raw_basis = bessel_basis(r, n=n, r_cut=r_cut)
envelope = cosine_cutoff(r, r_cut)
cut_basis = envelope * raw_basis

print("raw Bessel values:     ", raw_basis[:6])
print("cutoff envelope:      ", envelope[:6])
print("after multiplication: ", cut_basis[:6])
print(f"value at r_cut = {r_cut}:", cut_basis[-1])

assert torch.isclose(envelope[0], torch.tensor(1.0))
assert torch.isclose(cut_basis[-1], torch.tensor(0.0))

cutoff_probe = torch.tensor(r_cut, requires_grad=True)
cutoff_value = cosine_cutoff(cutoff_probe, r_cut) * bessel_basis(cutoff_probe, n, r_cut)
cutoff_slope = torch.autograd.grad(cutoff_value, cutoff_probe)[0]
assert torch.isclose(cutoff_value, torch.tensor(0.0))
assert torch.isclose(cutoff_slope, torch.tensor(0.0))
print("cutoff value and slope:", cutoff_value.item(), cutoff_slope.item())

plt.figure(figsize=(8, 4))
plt.plot(r, raw_basis, label="Raw Bessel basis $R_1(r)", linewidth=2)
plt.plot(r, cut_basis, label=r"Enveloped basis $f_{\mathrm{cut}}(r)R_1(r)$", linewidth=2)
plt.axvline(r_cut, color="black", linestyle="--", alpha=0.5, label=r"$r_{\mathrm{cut}}$")
plt.xlabel("Distance $r$")
plt.ylabel("Feature value")
plt.title("Question: does the edge feature reach the cutoff smoothly?")
plt.grid(alpha=0.2)
plt.legend()
plt.show()
print("Takeaway: both the enveloped feature and its slope are zero at the cutoff.")

### Why must the cutoff arrive with zero slope?

The plot makes the practical reason visible. The orange curve does not merely *reach* zero at $r_\mathrm{cut}$; it arrives flat. In other words, both the feature and its first derivative vanish at the boundary:

$$\widetilde R_n(r_\mathrm{cut})=0, \qquad \frac{d\widetilde R_n}{dr}(r_\mathrm{cut})=0.$$

MACE predicts an energy $E$, and forces are obtained by differentiating that energy with respect to atomic positions:

$$\mathbf F_i=-\nabla_{\mathbf r_i}E.$$

Imagine a neighbour atom moving outward and crossing the cutoff radius. If its edge feature were switched off abruptly, the predicted energy could have a kink. Differentiating a kink produces a sudden, artificial change in force—an unphysical kick caused by the model boundary rather than by chemistry. A smooth, zero-slope envelope lets the neighbour fade away gently: its contribution to both energy and force is already disappearing before the edge is removed.

### From fixed basis curves to a learned distance feature

At every distance $r$, the three cutoff basis functions give a three-number encoding of that distance. A learned weighted sum turns that encoding into one scalar radial feature:

$$\phi(r)=\sum_n a_n\widetilde R_n(r).$$

The coefficients $a_n$ below are deliberately chosen example values, not trained parameters. Their only purpose is to show that a neural network can recombine the same fixed basis functions into many different distance-dependent shapes.

In [ ]:
# Three fixed radial measurements at each of the 400 distances.
radial_basis_by_mode = torch.stack([
    cosine_cutoff(r, r_cut) * bessel_basis(r, n, r_cut)
    for n in [1, 2, 3]
])  # (P=3, S=400), convenient for plotting one curve per row
radial_features = radial_basis_by_mode.T  # (S=400, P=3), one feature row per distance

# Illustrative learned coefficients: in a real model, training would set them.
radial_weights = torch.tensor([0.7, -0.35, 0.5])
learned_radial_feature = radial_features @ radial_weights

print("radial feature shape: ", radial_features.shape)
print("learned feature shape:", learned_radial_feature.shape)

plt.figure(figsize=(8, 4))
for n, curve in enumerate(radial_basis_by_mode, start=1):
    plt.plot(r, curve, color="gray", alpha=0.35, label=f"$\\widetilde R_{n}(r)$")
plt.plot(r, learned_radial_feature, color="crimson", linewidth=3, label="Learned mixture $\\phi(r)$")
plt.axvline(r_cut, color="black", linestyle="--", alpha=0.5)
plt.xlabel("Distance $r$")
plt.ylabel("Feature value")
plt.title("Question: how can learned weights reshape a fixed radial encoding?")
plt.legend()
plt.grid(alpha=0.2)
plt.show()
print("Takeaway: learning combines fixed modes; it does not change their scalar rotation type.")

# Part II — assemble MACE

The remaining sections compose the primitives already tested: distance, direction, channel mixing, tensor products, and Clebsch–Gordan projection. The notation is dense, so we first choose a storage convention that can actually be implemented.

## Global type and shape spine

An order-$l$ block has $2l+1$ components. Because this width changes with $l$, we do **not** pretend that all orders fit a dense `[..., l, m]` rectangle. We store one tensor per angular order:

| Object | Realizable storage for one fixed order | Meaning |
|---|---|---|
| edge displacement $\mathbf r_{ji}$ | `(E, 3)` | one Cartesian vector per directed edge |
| edge distance $r_{ji}$ | `(E,)` | invariant scalar per edge |
| edge direction $\hat{\mathbf r}_{ji}$ | `(E, 3)` | unit vector per edge |
| radial basis $\widetilde R_n(r_{ji})$ | `(E, P)` | `P` invariant distance measurements per edge |
| harmonics $Y^{(l)}(\hat r_{ji})$ | `(E, 2*l+1)` | one order-$l$ angular block per edge |
| hidden block `h[l]` | `(N, K_in[l], 2*l+1)` | node, channel $\tilde k$, component $m$ |
| pooled block `A[l]` | `(N, K_A[l], 2*l+1)` | node, channel $k$, component $m$ |
| product block `B[L]` | `(N, P_L, K_B[L], 2*L+1)` | node, path, channel, output component $M$ |
| message block `message[L]` | `(N, K_B[L], 2*L+1)` | compatible paths mixed, path axis removed |

Here `E` is the number of directed edges, `N` the number of atoms, `P` the number of radial basis functions, and `K_*` feature-channel counts. `P_L` enumerates legal coupling paths ending at output order $L$. In equations, $l,m$ name an input rotation type and its component; $L,M$ name the final output type and component. Channel $k$ labels learned copies of a type. Correlation order $\nu$ counts multiplied $A$ factors; in this atom-centred construction it corresponds to body order $\nu+1$ when the centre atom is included. The label $\eta_\nu$ identifies a coupling path; it is not a component, a channel, or a learned value.

> **Implementation rule:** fix $l$ or $L$ first, then the final component axis has a known size. Every code example below follows this per-order convention.

## 15. Eq. 8 — construct the edge-aware feature $A$

$$\boxed{\begin{aligned}
A^{(t)}_{i,k l_3 m_3} &= \sum_{l_1m_1,\,l_2m_2} C^{l_3m_3}_{l_1m_1,\,l_2m_2}\sum_{j\in\mathcal N(i)} R^{(t)}_{k l_1 l_2 l_3}(r_{ji})\,Y^{m_1}_{l_1}(\hat r_{ji})\sum_{\tilde k} W^{(t)}_{k\tilde k l_2}h^{(t)}_{j,\tilde k l_2m_2}.
\end{aligned}}$$

### Eq. 8 type ledger

Read one edge $j\to i$ before reading the sums. Layer $t$ is fixed. Atom $i$ receives; $j\in\mathcal N(i)$ sends. For selected input orders $l_1,l_2$ and legal output order $l_3$, the relevant per-order objects are:

| Symbol | One-edge shape | Role |
|---|---:|---|
| $R^{(t)}_{k l_1l_2l_3}(r_{ji})$ | `(K_out,)` | learned invariant radial coefficient for each output channel $k$ |
| $Y^{(l_1)}(\hat r_{ji})$ | `(2*l1+1,)` | directional block, component $m_1$ |
| $h_j^{(t,l_2)}$ | `(K_in, 2*l2+1)` | sender hidden block, channel $\tilde k$, component $m_2$ |
| $W^{(t,l_2)}$ | `(K_out, K_in)` | learned channel mixing, shared over every $m_2$ |
| $C_{l_1l_2\to l_3}$ | `(2*l3+1, (2*l1+1)*(2*l2+1))` | fixed CG projection |
| edge contribution to `A[l3]` | `(K_out, 2*l3+1)` | output channel $k$, component $m_3$ |

The paper's radial function uses the letter $R$, but it is **not** the $3\times3$ rotation matrix from Section 3. It returns scalars and therefore cannot change an angular type. The sums simply repeat this typed one-edge operation over input orders and neighbours.

Equation 8 is the moment when the ingredients we built separately finally meet. For every neighbour $j$ of a centre atom $i$, MACE takes three things:

1. **How far away is $j$?** The radial network supplies a scalar coefficient depending on $r_{ji}$.
2. **Which direction is $j$?** A spherical harmonic $Y_{l_1}^{m_1}(\hat r_{ji})$ supplies an $l_1$-type directional pattern.
3. **What has $j$ already learned about its environment?** Its hidden feature $h_{j,\tilde{k}l_2m_2}^{(t)}$ has rotation type $l_2$.

The radial coefficient scales the directional pattern; that pattern is then tensor-multiplied with the neighbour feature. Clebsch–Gordan coefficients perform the fixed, symmetry-safe regrouping:

$$l_1\otimes l_2 \longrightarrow l_3,$$

and the result is accumulated over all neighbours. The output is $A_{i,k l_3m_3}^{(t)}$: a feature located at atom $i$, in learned channel $k$, with rotation type $l_3$ and component $m_3$.

### Read Eq. 8 from the inside out

For one edge $j\to i$, start with a direction feature $Y^{(l_1)}(\hat r_{ji})$ and a neighbour feature $h_j^{(l_2)}$. Their product initially has $(2l_1+1)(2l_2+1)$ entries. The Clebsch–Gordan map reorganises those entries into legal output blocks $l_3=|l_1-l_2|,\ldots,l_1+l_2$. The radial function and learned channel weights decide how strongly that particular edge contributes; they do **not** alter its rotation type. Finally, summing edge contributions over $j\in\mathcal N(i)$ pools the local neighbourhood without losing equivariance.

In one line: **Equation 8 combines directional edge information with a neighbour's learned content, weights that contribution by edge distance, couples it into a symmetry-valid output, and then pools the result at the centre atom $i$.**

The next cell turns exactly that sentence into a tiny calculation.

In [ ]:
# MACE LAB A — one-edge Eq. 8 with l1=1, l2=1, K_in=2, K_out=1.
eq8_edge_distance = torch.tensor([1.5])                         # (E=1,)
eq8_edge_direction = torch.tensor([[1.0, 1.0, 0.0]])            # (E=1, 3)
eq8_edge_direction = eq8_edge_direction / torch.linalg.vector_norm(
    eq8_edge_direction, dim=-1, keepdim=True
)
eq8_neighbor_h_l1 = torch.tensor([                              # (E=1, K_in=2, m2=3)
    [[0.20, -0.10, 0.40], [0.30, 0.50, -0.20]]
])
eq8_channel_map = torch.tensor([[0.60, -0.25]])                 # (K_out=1, K_in=2)

# 1. Invariant radial row -> a distinct coefficient for each edge, channel, and output L.
eq8_radial_features = torch.stack([
    cosine_cutoff(eq8_edge_distance, r_cut)
    * bessel_basis(eq8_edge_distance, mode, r_cut)
    for mode in [1, 2, 3]
], dim=-1)                                                      # (E=1, P=3)
eq8_output_orders = (0, 1, 2)
eq8_radial_map = torch.tensor([                                # (P=3, K_out=1, output_L=3)
    [[ 0.40, -0.10,  0.25]],
    [[-0.20,  0.30,  0.15]],
    [[ 0.10,  0.05, -0.35]],
])
eq8_radial_coefficient = torch.einsum(
    "ep,pko->eko", eq8_radial_features, eq8_radial_map
)                                                               # (E=1, K_out=1, output_L=3)

# 2. Angular edge block and sender-channel mixing. The map never touches m2.
eq8_Y_l1 = generate_spherical_harmonics(1, eq8_edge_direction) # (E=1, m1=3)
eq8_mixed_h_l1 = torch.einsum(
    "ok,ekm->eom", eq8_channel_map, eq8_neighbor_h_l1
)                                                               # (E=1, K_out=1, m2=3)

# 3. Raw tensor product over m1,m2, then fixed CG-style change of basis.
eq8_raw_product = torch.einsum(
    "em,eon->eomn", eq8_Y_l1, eq8_mixed_h_l1
)                                                               # (E=1, K_out=1, 3, 3)
eq8_raw_flat = eq8_raw_product.flatten(start_dim=2)             # (E=1, K_out=1, 9)
eq8_coupled = torch.einsum("ab,eob->eoa", C_cart, eq8_raw_flat)  # 1+3+5

# 4. Split by output L, apply that L's radial coefficient, then pool over E.
eq8_coupled_L0 = eq8_coupled[:, :, :1]
eq8_coupled_L1 = eq8_coupled[:, :, 1:4]
eq8_coupled_L2 = eq8_coupled[:, :, 4:9]
eq8_weighted_L0 = eq8_radial_coefficient[:, :, 0, None] * eq8_coupled_L0
eq8_weighted_L1 = eq8_radial_coefficient[:, :, 1, None] * eq8_coupled_L1
eq8_weighted_L2 = eq8_radial_coefficient[:, :, 2, None] * eq8_coupled_L2
A_eq8_L0 = eq8_weighted_L0.sum(dim=0)                           # (K_out=1, M=1)
A_eq8_L1 = eq8_weighted_L1.sum(dim=0)                           # (K_out=1, M=3)
A_eq8_L2 = eq8_weighted_L2.sum(dim=0)                           # (K_out=1, M=5)
eq8_weighted_all_orders = torch.cat(
    [eq8_weighted_L0, eq8_weighted_L1, eq8_weighted_L2], dim=-1
)                                                               # diagnostic concatenation (E,K_out,9)

print("radial features (E, P):       ", eq8_radial_features.shape)
print("radial coefficients (E,K,L): ", eq8_radial_coefficient.shape)
print("harmonics (E, m1):            ", eq8_Y_l1.shape)
print("mixed hidden (E, K_out, m2):  ", eq8_mixed_h_l1.shape)
print("raw product (E, K_out, m1,m2):", eq8_raw_product.shape)
print("A blocks L=0,1,2:             ", A_eq8_L0.shape, A_eq8_L1.shape, A_eq8_L2.shape)

assert torch.allclose(
    torch.einsum("ab,eoa->eob", C_cart, eq8_coupled), eq8_raw_flat
)
assert A_eq8_L0.shape == (1, 1)
assert A_eq8_L1.shape == (1, 3)
assert A_eq8_L2.shape == (1, 5)

# Equivariance check: rotate both l=1 inputs, then compare in coupled coordinates.
eq8_rotated_direction = eq8_edge_direction @ R.T
eq8_rotated_hidden = eq8_neighbor_h_l1 @ R.T
eq8_Y_l1_rotated = generate_spherical_harmonics(1, eq8_rotated_direction)
eq8_mixed_h_rotated = torch.einsum("ok,ekm->eom", eq8_channel_map, eq8_rotated_hidden)
eq8_raw_rotated = torch.einsum(
    "em,eon->eomn", eq8_Y_l1_rotated, eq8_mixed_h_rotated
).flatten(start_dim=2)
eq8_coupled_rotated = torch.einsum("ab,eob->eoa", C_cart, eq8_raw_rotated)
coupled_rotation = C_cart @ torch.kron(R, R) @ C_cart.T
eq8_weighted_rotated = torch.cat([
    eq8_radial_coefficient[:, :, 0, None] * eq8_coupled_rotated[:, :, :1],
    eq8_radial_coefficient[:, :, 1, None] * eq8_coupled_rotated[:, :, 1:4],
    eq8_radial_coefficient[:, :, 2, None] * eq8_coupled_rotated[:, :, 4:9],
], dim=-1)
eq8_weighted_expected = eq8_weighted_all_orders @ coupled_rotation.T
eq8_equivariance_error = torch.max(torch.abs(eq8_weighted_rotated - eq8_weighted_expected))
assert torch.allclose(eq8_weighted_rotated, eq8_weighted_expected, atol=1e-5)
print("Eq. 8 full weighted equivariance error:", eq8_equivariance_error.item())

## 16. Eq. 9 — the first-layer message

$$\boxed{
A^{(1)}_{i,k l_1m_1}=\sum_{j\in\mathcal N(i)}R^{(1)}_{k l_1}(r_{ji})\,Y^{m_1}_{l_1}(\hat r_{ji})\,W^{(1)}_{k z_j}.
}$$

Equation 9 is Equation 8 at the very first layer. At this point, atom $j$ has not yet learned a geometric hidden feature. Its only input is its chemical identity $z_j$—for example, hydrogen, carbon, or oxygen. Chemical identity does not rotate, so this initial feature is a scalar: $l_2=0$.

That one fact causes the simplification:

$$l_1\otimes 0=l_1.$$

There is no nontrivial tensor product to reorganise, so the Clebsch–Gordan coupling and the $l_2,m_2,\tilde k$ bookkeeping vanish. The element-dependent parameter $W^{(1)}_{kz_j}$ selects an embedding for neighbour $j$'s species. The learned radial function $R^{(1)}_{kl_1}(r_{ji})$ says how strongly that species contributes at this distance. The spherical harmonic $Y^{m_1}_{l_1}(\hat r_{ji})$ supplies the direction.

For each neighbour, Eq. 9 therefore creates an edge feature: **species-dependent strength × distance-dependent strength × directional pattern**. Summing those edge features gives atom $i$ its first equivariant description of the surrounding chemical environment.

### A concrete two-neighbour picture

Suppose centre atom $i$ has two neighbours. In our toy calculation, row 0 is a hydrogen neighbour lying along $+\hat x$, and row 1 is an oxygen neighbour lying along $+\hat y$:

$$
\hat r_{1i}=[1,0,0] \longleftrightarrow j=1=\mathrm{H},
\qquad
\hat r_{2i}=[0,1,0] \longleftrightarrow j=2=\mathrm{O}.
$$

The first row of the direction tensor, the first row of the spherical-harmonic tensor, and the first species lookup must all refer to that same hydrogen neighbour. The second rows similarly all refer to oxygen. Notice that $\mathrm H$ means the chemical element hydrogen; it is not the hidden feature $h$. At the first layer, no geometric hidden feature exists yet—the model begins with only the element label $z_j$.

This is why the first MACE layer is computationally cheap: it creates directional features from scalar element labels, without yet having to couple two non-scalar irreps.

In [ ]:
eq9_direction_H = torch.tensor([[1.0, 0.0, 0.0]])
eq9_direction_O = torch.tensor([[0.0, 1.0, 0.0]])
eq9_edge_directions = torch.cat([eq9_direction_H, eq9_direction_O], dim=0)
assert eq9_edge_directions.shape == (2, 3)
assert eq9_edge_directions.is_floating_point()

In [ ]:
eq9_species_ids = torch.tensor([0, 1])  # row 0 = H, row 1 = O
eq9_species_embedding = torch.tensor([[0.01, 0.12]])  # (K_out=1, Z=2)
eq9_species_weights = eq9_species_embedding[:, eq9_species_ids].T  # (E=2, K_out=1)
print("species lookup shape (E, K_out):", eq9_species_weights.shape)

In [ ]:
# MACE LAB B — Eq. 9 as a special case of Eq. 8
# Step 1: learned radial strength for two edges.
# Row 0 is H along +x; row 1 is O along +y.
edge_lengths = torch.tensor([1.0, 2.0])

edge_radial_features = torch.stack([
    cosine_cutoff(edge_lengths, r_cut) * bessel_basis(edge_lengths, n, r_cut)
    for n in [1, 2, 3]
], dim=-1)  # (E=2, P=3)
radial_strength = edge_radial_features @ radial_weights  # (E=2,)

print("basis shape (E, P):   ", edge_radial_features.shape)
print("radial-strength shape:", radial_strength.shape)
print("one value per edge:   ", radial_strength)

In [ ]:
edge_strength = eq9_species_weights * radial_strength[:, None]  # (E=2, K_out=1)
print("edge strength shape (E, K_out):", edge_strength.shape)

In [ ]:
Y_l1 = generate_spherical_harmonics(1, eq9_edge_directions)  # (E=2, m=3)
print("Y_l1 shape (E, m):", Y_l1.shape)
print(Y_l1)

In [ ]:
edge_messages = torch.einsum("ek,em->ekm", edge_strength, Y_l1)  # (E,K_out,m)
print("edge message shape (E, K_out, m):", edge_messages.shape)
print(edge_messages)

In [ ]:
A_i_l1 = edge_messages.sum(dim=0)  # (K_out=1, m=3)

# Full Eq. 8 route when the sender hidden feature is a scalar one-hot species vector.
eq9_species_one_hot = torch.nn.functional.one_hot(
    eq9_species_ids, num_classes=2
).to(dtype=eq9_edge_directions.dtype)                            # (E=2, K_in=2)
eq8_mixed_scalar_h = torch.einsum(
    "ok,ek->eo", eq9_species_embedding, eq9_species_one_hot
)                                                               # (E=2, K_out=1)
eq8_scalar_hidden_messages = torch.einsum(
    "e,ek,em->ekm", radial_strength, eq8_mixed_scalar_h, Y_l1
)                                                               # l1 x 0 -> l1; CG map is identity
A_i_l1_from_eq8 = eq8_scalar_hidden_messages.sum(dim=0)
assert torch.allclose(A_i_l1_from_eq8, A_i_l1)

# Rotation check for the pooled L=1 block.
eq9_rotated_directions = eq9_edge_directions @ R.T
eq9_Y_l1_rotated = generate_spherical_harmonics(1, eq9_rotated_directions)
A_i_l1_from_rotated_input = torch.einsum(
    "ek,em->km", edge_strength, eq9_Y_l1_rotated
)
A_i_l1_then_rotated = A_i_l1 @ D[1].T
eq9_equivariance_error = torch.max(
    torch.abs(A_i_l1_from_rotated_input - A_i_l1_then_rotated)
)
assert torch.allclose(A_i_l1_from_rotated_input, A_i_l1_then_rotated, atol=1e-5)
print("Eq. 8 scalar route equals Eq. 9:", torch.allclose(A_i_l1_from_eq8, A_i_l1))
print("Eq. 9 pooled shape (K_out, m):", A_i_l1.shape)
print("Eq. 9 equivariance error:", eq9_equivariance_error.item())
A_i_l1

### Each neighbour now has a complete Eq. 9 message

`edge_strength` has shape `(E=2, K_out=1)`: one scalar for each edge and output channel. `Y_l1` has shape `(E=2, m=3)`: each edge has three $l=1$ components. The explicit `einsum` keeps the edge, channel, and angular-component axes separate:

$$
\texttt{edge\_messages}_{j,k,m}=\underbrace{W_{kz_j}}_{\text{species}}\;\underbrace{R_{kl_1}(r_{ji})}_{\text{distance}}\;\underbrace{Y^{m}_{l_1}(\hat r_{ji})}_{\text{direction}}.
$$

The result has shape `(E=2, K_out=1, m=3)`. Edge 0 is the complete vector-like message sent by hydrogen along $+\hat x$; edge 1 is the message sent by oxygen along $+\hat y$. Summing only the edge axis produces `A_i_l1` with shape `(K_out=1, m=3)`. The cell also constructs the scalar one-hot sender feature used by full Eq. 8 and verifies that this route equals the simplified Eq. 9 route.

> **Takeaway:** Eq. 9 is not a different mechanism. It is Eq. 8 with scalar species input, so $l_1\otimes0\to l_1$ and the CG projection becomes the identity.

### Paper-notation checkpoint — generalized CG indices

Map the concrete CG lab above onto the paper's generalized coefficient notation.

Identify the input tuple, output `(L, M)`, and coupling-path label `η_ν`. Record which quantities are indices, fixed coefficients, and learned parameters.

For fixed correlation order $\nu$:

| Notation | Kind | Meaning |
|---|---|---|
| $lm=((l_1,m_1),\ldots,(l_\nu,m_\nu))$ | summed input indices | one angular order/component pair per factor slot $\xi$ |
| $(L,M)$ | output indices | final order and its component |
| $\eta_\nu$ | discrete path label | legal sequence of intermediate couplings; not itself an angular order |
| $C^{LM}_{\eta_\nu,lm}$ | fixed coefficient | generalized CG map determined by symmetry and basis convention |
| $w_{k\tilde k l_\xi}$ | learned parameter | mixes input channels $\tilde k$ into output channel $k$, shared across $m_\xi$ |

The tuple $lm$ is shorthand, not one index with a rectangular fixed size. In code we select the orders $(l_1,\ldots,l_\nu)$ first; each factor then has its own component width $2l_\xi+1$.

In [ ]:
# PAPER CHECKPOINT — executable labels matching the tree in the next section.
paper_index_example = {
    "factor_slots_xi": (1, 2, 3),
    "input_orders": factor_orders_nu3,
    "input_component_widths": tuple(2 * l_order + 1 for l_order in factor_orders_nu3),
    "final_order_L": target_L_nu3,
    "final_component_width": 2 * target_L_nu3 + 1,
    "eta3_labels": tuple(eta3_paths),
}
assert paper_index_example["eta3_labels"] == ("p_0", "p_1", "p_2")
print(paper_index_example)
print("eta_3 labels complete routes; l12 is stored separately inside each route.")

## 17. Eq. 10 — build higher-body features

$$\boxed{
B^{(t)}_{i,\eta_\nu kLM}=\sum_{lm}C^{LM}_{\eta_\nu,lm}\prod_{\xi=1}^{\nu}\sum_{\tilde k}w^{(t)}_{k\tilde k l_\xi}A^{(t)}_{i,\tilde k l_\xi m_\xi},
\qquad lm=(l_1m_1,\ldots,l_\nu m_\nu).
}$$

Equation 9 gave us a pooled two-body feature $A_i$: a symmetry-aware summary of individual neighbours around atom $i$. Equation 10 is MACE's key many-body step. It combines several $A_i$ features **at the same centre atom**, allowing the model to describe correlations among several neighbours without explicitly looping over every triplet or quadruplet.

### Read the equation from the inside out

For each factor $\xi$, first mix the learned channels of one $A_i$ block using $w^{(t)}_{k\tilde k l_\xi}$. This is ordinary channel mixing: it may change which feature channel we use, but it leaves that factor's rotation type $l_\xi$ untouched.

Next, multiply $\nu$ such channel-mixed $A_i$ blocks. For $\nu=2$, this is the familiar tensor product of two features; for $\nu=3$, it is a three-factor product. The raw product contains all combinations of their $m$-components.

Finally, the fixed generalised Clebsch–Gordan coefficients $C^{LM}_{\eta_\nu,lm}$ regroup that raw product into a valid output block of rotation type $L$, component $M$. The path label $\eta_\nu$ records *which legal sequence of couplings* was used when several sequences can produce the same final $L$.

In one line: **Eq. 10 takes several already-pooled two-body views of the same local environment, multiplies them, and symmetry-couples the result into a higher-body feature $B_i$.**

This is the computational trick behind MACE: it gets higher-body information from products of pooled features, instead of enumerating every neighbour triplet or quadruplet.

In [ ]:
raw_two_body_product = torch.outer(A_i_l1[0], A_i_l1[0])
print(raw_two_body_product)
print(raw_two_body_product.shape)

### Why the path label $\eta_3$ matters

For three vector-like inputs and a final vector-like output, $1\otimes1\otimes1\to L=1$, the same final rotation type can be reached through three different intermediate orders:

```text
factor ξ=1: (l1=1,m1) ─┐
                        ├─ couple → l12 in {0,1,2} ─┐
factor ξ=2: (l2=1,m2) ─┘                            ├─ couple → (L=1,M)
factor ξ=3: (l3=1,m3) ─────────────────────────────┘
```

| Path label | Intermediate order | Coupling route |
| --- | ---: | --- |
| $\eta_3=p_0$ | $l_{12}=0$ | $((1\otimes1)_0\otimes1)_1$ |
| $\eta_3=p_1$ | $l_{12}=1$ | $((1\otimes1)_1\otimes1)_1$ |
| $\eta_3=p_2$ | $l_{12}=2$ | $((1\otimes1)_2\otimes1)_1$ |

The subscript in $\eta_3$ means *three input factors*. The symbols $p_0,p_1,p_2$ are arbitrary names for complete routes; the separate column records the intermediate angular order $l_{12}$. In this tiny tree, $l_{12}$ happens to distinguish the paths, but a general $\eta_\nu$ may need several intermediate orders. All three outputs have final order $L=1$, yet remain separate until Eq. 11 mixes the path axis.

In [ ]:
# MACE LAB C — prepare two independent l=1 input channels.
# Rows: input channels k_tilde=0,1. Columns: species H,O (one edge of each here).
species_embeddings_by_channel = torch.tensor([
    [0.01,  0.12],  # channel k=0
    [0.08, -0.03],  # channel k=1
])

# Eq. 9 again, now producing two independent l=1 A-blocks.
edge_strengths_by_channel = species_embeddings_by_channel * radial_strength.unsqueeze(0)
print(edge_strengths_by_channel.shape)
print(edge_strengths_by_channel)


In [ ]:
A_channels_l1 = (edge_strengths_by_channel.unsqueeze(-1) * Y_l1.unsqueeze(0)).sum(dim=1)
print(A_channels_l1.shape)


In [ ]:
A_k0_l1, A_k1_l1 = A_channels_l1

# Diagnostic only: a generic cross-channel tensor product can have a nonzero L=1 block.
# The next cell contrasts this with the tied-channel structure of the printed Eq. 10.
raw_cross_product = torch.outer(A_k0_l1, A_k1_l1)
cross_channel_coupled = C_cart @ raw_cross_product.flatten()
cross_channel_L1 = cross_channel_coupled[1:4]
print("cross-channel raw shape:", raw_cross_product.shape)
print("diagnostic nonzero L=1:", cross_channel_L1)
assert torch.linalg.vector_norm(cross_channel_L1) > 0
assert torch.allclose(C_cart.T @ cross_channel_coupled, raw_cross_product.flatten())


In [ ]:
# Literal Eq. 10 for nu=2, l1=l2=1, one output channel k.
# The same learned channel mixture w[k,k_tilde,l=1] is used in both factor slots.
eq10_channel_weights_l1 = torch.tensor([[0.60, -0.25]])       # (K_out=1, K_in=2)
eq10_factor_l1 = torch.einsum(
    "ok,km->om", eq10_channel_weights_l1, A_channels_l1
)                                                             # (K_out=1, m=3)
eq10_raw_product = torch.einsum(
    "km,kn->kmn", eq10_factor_l1, eq10_factor_l1
)                                                             # (K_out=1, m1=3, m2=3)
eq10_raw_flat = eq10_raw_product.flatten(start_dim=1)         # (K_out=1, 9)
eq10_coupled = torch.einsum("ab,kb->ka", C_cart, eq10_raw_flat)

# Keep an explicit singleton eta_2 path axis: (P_L=1, K_out=1, 2L+1).
eta2_labels = ("single_path",)
B_nu2_L0 = eq10_coupled[:, :1].unsqueeze(0)
B_nu2_L1 = eq10_coupled[:, 1:4].unsqueeze(0)
B_nu2_L2 = eq10_coupled[:, 4:9].unsqueeze(0)

print("A input blocks (K_in, m):      ", A_channels_l1.shape)
print("mixed factor (K_out, m):       ", eq10_factor_l1.shape)
print("raw product (K_out, m1, m2):  ", eq10_raw_product.shape)
print("B[L=0] (path, K_out, M):      ", B_nu2_L0.shape)
print("B[L=1] (path, K_out, M):      ", B_nu2_L1.shape)
print("B[L=2] (path, K_out, M):      ", B_nu2_L2.shape)

assert torch.allclose(
    torch.einsum("ab,ka->kb", C_cart, eq10_coupled), eq10_raw_flat
)
assert torch.allclose(B_nu2_L1, torch.zeros_like(B_nu2_L1), atol=1e-8)

### What the cross-channel comparison teaches—and what Eq. 10 actually does

The diagnostic cell preserves an important geometric observation: two different $l=1$ channel blocks are generally not parallel, so their generic cross-product tensor can contain $L=0$, $L=1$, and $L=2$ sectors. Its nonzero $L=1$ block proves that the antisymmetric sector was absent from $A\otimes A$ because the two factors were identical, not because CG coupling forbids $L=1$.

However, that cross-channel diagnostic is **not** the literal printed Eq. 10. In Eq. 10, a fixed output channel $k$ uses the same channel map $w_{k\tilde k,l=1}$ in both identical $l=1$ factor slots. The staged implementation therefore first mixes the two input channels into one `eq10_factor_l1[k]`, then forms that factor's self-product. Its $L=1$ block correctly vanishes, while its $L=0$ and $L=2$ blocks remain.

For the selected two-factor route, $\nu=2$ is the **correlation order**, corresponding here to body order $3$ after including centre atom $i$. The path axis $\eta_2$ is a singleton for each selected $L$, but the code retains it explicitly so Eq. 11 has a real path axis to mix.

> **Takeaway:** channels $k$ are learned copies; factor slots $\xi$ are multiplied; $m$ components are CG-coupled. Treating those axes as interchangeable changes the equation.

## 18. Eq. 11 — form the message

$$\boxed{
m^{(t)}_{i,kLM}=\sum_{\nu}\sum_{\eta_\nu}W^{(t)}_{z_i kL,\eta_\nu}B^{(t)}_{i,\eta_\nu kLM}.
}$$

Equation 10 made a collection of higher-body features $B_i$. Equation 11 is the final assembly step: it takes compatible $B_i$ blocks and adds them into the message $m_i$ that will update atom $i$.

The word **compatible** is doing important work. The learned weight $W^{(t)}_{z_i kL,\eta_\nu}$ is a scalar, so it can rescale a whole irrep block but cannot change its rotation type. Therefore, when forming an output with order $L$, Eq. 11 only adds $B$ blocks that already have that same order $L$ and the same component index $M$.

For example, an $L=1$ message has three $M$-components. MACE may combine any nonzero $L=1$ blocks produced by different correlation orders or coupling paths. It may **not** add an $L=2$ five-component block: that would mix different rotation behaviours. In the concrete lab below we use $L=0$, because the literal $\nu=2$, $l_1=l_2=1$ Eq. 10 example has nonzero scalar and quadrupole sectors but a zero $L=1$ sector.

The receiving atom's element $z_i$ appears in the weight. This lets the model use different learned mixtures when the centre atom is carbon versus oxygen, while retaining the same symmetry rule for every species.

In one line: **Eq. 11 selects and linearly combines all higher-body features that transform in the same way, producing the final equivariant message at atom $i$.**

The pair $(\nu,\eta_\nu)$ therefore behaves like path metadata. Eq. 11 removes that path axis by a learned, receiver-species-dependent weighted sum, but only after $L$ has been fixed.

In [ ]:
# MACE LAB D — Eq. 11 at fixed L=0, with explicit path and species axes.
Y_l0 = generate_spherical_harmonics(0, eq9_edge_directions)       # (E=2, M=1)
A_i_l0 = torch.einsum("ek,em->km", edge_strength, Y_l0)       # (K_out=1, M=1)
B_nu1_L0 = A_i_l0.unsqueeze(0)                                  # (path=1, K=1, M=1)
# B_nu2_L0 above already has shape (path=1, K=1, M=1).

B_compatible_L0 = torch.cat([B_nu1_L0, B_nu2_L0], dim=0)        # (all_paths=2,K=1,M=1)
eq11_path_metadata = ((1, "identity"), (2, eta2_labels[0]))

# W[z_i, k, path] after fixing L=0. Rows are H, O, C receiver species.
eq11_weights_by_species_L0 = torch.tensor([
    [[0.10, -0.20]],
    [[0.25,  0.15]],
    [[0.35, -0.80]],
])                                                             # (Z=3, K=1, all_paths=2)
receiver_species_id = 2                                         # carbon-like centre
eq11_weights_L0 = eq11_weights_by_species_L0[receiver_species_id].T  # (path=2,K=1)
message_i_L0 = torch.einsum("pk,pkm->km", eq11_weights_L0, B_compatible_L0)

print("path metadata (nu, eta_nu):     ", eq11_path_metadata)
print("compatible B (path, K, M):     ", B_compatible_L0.shape)
print("receiver weights (path, K):    ", eq11_weights_L0.shape)
print("final message (K, M), L=0:    ", message_i_L0.shape)
print("final scalar-like message:     ", message_i_L0)
assert message_i_L0.shape == (1, 1)

### Eq. 11 in the lab

The lab fixes $L=0$, so every candidate has one $M=0$ component. `B_compatible_L0` keeps two explicit paths: the identity path at correlation order $\nu=1$ (body order $2$) and the singleton path at correlation order $\nu=2$ (body order $3$). The lookup `W[z_i,k,path]` selects carbon-receiver weights, and the `einsum` sums only the path axis. The channel and component axes survive in `message_i_L0`.

The same operation works for $L=1$ or $L=2$ when compatible nonzero blocks exist: the scalar path weight is shared across all $2L+1$ values of $M$.

> **Takeaway:** Eq. 11 learns which body orders and coupling paths matter; it never learns how an $L$ block rotates.

### Eq. 10–11: the complete local message story

We can now read the two equations as one pipeline. Equation 9 creates an $A_i$ block by collecting neighbour messages at the centre atom. Equation 10 takes $\nu$ of those blocks, forms their tensor product, and uses Clebsch–Gordan coefficients to split the result into legal irreducible blocks $B_{i,\eta_\nu kLM}$.

The number $\nu$ counts the factors in that product, so $\nu=2$ describes a three-body correlation when the centre atom is included. Equation 11 then gathers every **nonzero** $B$ block with the requested output order $L$. It applies a learned scalar weight to each compatible block and adds them. Compatibility means the same $L$ and the same $M$ layout—not the same correlation order. Blocks with different $L$ may not be added.

The implementation has therefore demonstrated the essential MACE pattern:

$$\text{neighbour messages}\;\longrightarrow\;A_i\;\longrightarrow\;A_i^{\otimes\nu}\;\longrightarrow\;B_{i,\eta_\nu kLM}\;\longrightarrow\;m_{i,kLM}.$$ 

The next step is no longer another toy tensor product. We move to the actual implementation ladder: represent these blocks with explicit channel/order/component axes, apply the radial and angular edge functions, and test equivariance after every stage.

## Minimal MACE implementation ladder

The equation labs operate on one centre. A real implementation adds node and edge axes but preserves the same typed operations. Build it in this order; do not advance until the decisive test passes.

| Step | Input → output | Symmetry contract | Smallest decisive test |
|---|---|---|---|
| 1. cutoff graph | positions `(N,3)`, species `(N,)` → `edge_index (2,E)` | translation/rotation invariant connectivity; relabelling atoms relabels edges | rigidly transform positions and compare the same edge pairs after undoing permutation |
| 2. relative geometry | positions, `edge_index` → displacement `(E,3)`, distance `(E,)`, direction `(E,3)` | displacement/direction equivariant; distance invariant | translate then rotate and compare both routes |
| 3. radial features | distance `(E,)` → `(E,P)` | invariant scalar features; zero value and slope at cutoff | rotate positions; radial rows unchanged; autograd cutoff check passes |
| 4. angular features | direction `(E,3)` → `Y[l] (E,2*l+1)` | order-$l$ equivariant | compare `Y[l](R r_hat)` with `D[l](R)Y[l](r_hat)` |
| 5. first-layer $A$ | graph, radial rows, `Y[l]`, species → `A[l] (N,K_l,2*l+1)` | node-permutation equivariant; neighbour sum order invariant; each $l$ block equivariant | permute edge rows and rotate geometry independently |
| 6. product basis $B$ | per-order `A` dict → `B[L] (N,P_L,K_L,2*L+1)` | correlation order $\nu$ and path $\eta_\nu$ explicit; output order $L$ equivariant | compare tensor-product/CG routes and check forbidden sectors vanish |
| 7. Eq. 11 message | `B[L]`, receiver species → `message[L] (N,K_L,2*L+1)` | mixes path/channel axes only; shared over $M$ | rotate each compatible path before versus after weighted sum |
| 8. node update | `h[L]`, `message[L]` → `h_next[L]` with the same shape | residual addition only between matching $L,M$ layouts | reject mismatched orders; verify equivariance of the update |
| 9. readout | `h_next[0] (N,K_0,1)` → atomic energy `(N,)` | invariant; uses scalar block only | rotate geometry; atomic predictions unchanged |
| 10. energy and forces | atomic energy → scalar $E$; positions → gradient `(N,3)` | total energy permutation/rotation invariant; force vector equivariant | compare $E(Rr)$ and $E(r)$, then verify $F(Rr)=RF(r)$ |

> **Build discipline:** shapes catch axis bugs; two-route tests catch symmetry bugs. You need both.

## 19. Implementation storage decision

### Directed-edge convention

For edge row `e`, set `edge_index[0,e] = i` (receiver/centre) and `edge_index[1,e] = j` (sender/neighbour). Define `displacement[e] = positions[j] - positions[i] = r_ji`, so the vector points from centre $i$ toward neighbour $j$. Eq. 9 edge messages are scatter-summed into receiver index $i$. Keep this convention fixed; reversing the displacement changes every odd-$l$ spherical-harmonic block.

Use the per-order storage convention introduced before Eq. 8:

- `A[l]` has shape `(N, K_in[l], 2*l+1)`;
- `channel_map[l]` has shape `(K_out[l], K_in[l])` and is shared across $m$;
- for selected input orders, `C[path, L]` maps the flattened factor-component axes to `(2*L+1)`;
- `B[L]` has shape `(N, P_L, K_out[L], 2*L+1)`;
- `message[L]` has shape `(N, K_out[L], 2*L+1)` after the path sum.

A production e3nn implementation may flatten these blocks into one `Irreps` axis. If you do that, write the slice belonging to every $l$ and keep the per-order view available in tests. Do not pad a fake global $m$ axis without documenting the mask.

### Scope of the executable block

The next cell implements one complete, differentiable educational path. Eq. 9 builds $A^{(0)}$ and $A^{(1)}$ blocks for every atom. Eq. 10 deliberately fixes correlation order $\nu=2$ and the input-order tuple $(l_1,l_2)=(1,1)$, so the explicit Cartesian CG matrix can expose the $L=0,1,2$ outputs. Eq. 11 keeps a singleton path axis, applies receiver-species weights, and removes only that axis. A scalar readout uses the updated $L=0$ block to produce atom energies, total energy, and forces.

This is enough to verify the architecture's mathematical spine without hiding it behind a framework abstraction. It is not a full MACE training implementation: generalized $\nu>2$ path enumeration, multiple interaction layers, batching, and learned modules remain production extensions rather than silent magic.

In [ ]:
# MINIMAL EDUCATIONAL MACE BLOCK
# This is a transparent one-layer implementation for the notebook, not production training code.

def build_cutoff_graph(positions, r_cut):
    """Return directed edges (receiver i, sender j) for 0 < r_ji < r_cut."""
    if positions.ndim != 2 or positions.shape[1] != 3:
        raise ValueError("positions must have shape (N, 3)")
    if r_cut <= 0:
        raise ValueError("r_cut must be positive")
    pair_displacements = positions[None, :, :] - positions[:, None, :]
    pair_distances = torch.linalg.vector_norm(pair_displacements, dim=-1)
    edge_mask = (pair_distances > 0) & (pair_distances < r_cut)
    receivers, senders = torch.where(edge_mask)
    return torch.stack([receivers, senders], dim=0)

def build_relative_geometry(positions, edge_index):
    """(N,3),(2,E) -> displacement (E,3), distance (E,), direction (E,3)."""
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError("edge_index must have shape (2, E)")
    receivers, senders = edge_index
    displacements = positions[senders] - positions[receivers]
    distances = torch.linalg.vector_norm(displacements, dim=-1)
    if torch.any(distances <= 0):
        raise ValueError("every directed edge must connect distinct positions")
    directions = displacements / distances[:, None]
    return displacements, distances, directions

def build_edge_features(distances, directions, max_l, radial_modes, r_cut):
    """Return invariant radial rows (E,P) and per-order harmonics Y[l]."""
    if directions.shape != (distances.shape[0], 3):
        raise ValueError("directions must have shape (E, 3)")
    radial_features = torch.stack([
        cosine_cutoff(distances, r_cut) * bessel_basis(distances, mode, r_cut)
        for mode in radial_modes
    ], dim=-1)
    harmonics_by_l = {
        l_order: generate_spherical_harmonics(l_order, directions)
        for l_order in range(max_l + 1)
    }
    return radial_features, harmonics_by_l

def build_first_layer_A(
    edge_index, radial_features, harmonics_by_l, species,
    radial_maps_by_l, species_embedding,
):
    """Implement Eq. 9 and return A[l] with shape (N,K_l,2*l+1)."""
    receivers, senders = edge_index
    num_nodes = species.shape[0]
    A_by_l = {}
    for l_order, Y_l in harmonics_by_l.items():
        radial_map = radial_maps_by_l[l_order].to(radial_features)       # (P,K_l)
        species_table = species_embedding.to(radial_features)           # (K,Z), shared over l
        radial_coefficients = radial_features @ radial_map             # (E,K_l)
        species_coefficients = species_table[:, species[senders]].T    # (E,K_l)
        edge_strengths = radial_coefficients * species_coefficients
        edge_messages = torch.einsum("ek,em->ekm", edge_strengths, Y_l)
        pooled = torch.zeros(
            num_nodes, edge_messages.shape[1], edge_messages.shape[2],
            dtype=edge_messages.dtype, device=edge_messages.device,
        ).index_add(0, receivers, edge_messages)
        A_by_l[l_order] = pooled
    return A_by_l

def build_product_basis(
    A_by_l, correlation_order, selected_order_tuple,
    channel_maps_by_l, cg_change_of_basis,
):
    """Educational Eq. 10 for nu=2 and (l1,l2)=(1,1)."""
    if correlation_order != 2 or tuple(selected_order_tuple) != (1, 1):
        raise ValueError(
            "this scoped lab accepts only nu=2 with (l1,l2)=(1,1)"
        )
    A_l1 = A_by_l[1]                                              # (N,K_in,3)
    channel_map = channel_maps_by_l[1].to(A_l1)                   # (K_out,K_in)
    factor_l1 = torch.einsum("ok,nkm->nom", channel_map, A_l1) # (N,K_out,3)
    raw_product = torch.einsum("nka,nkb->nkab", factor_l1, factor_l1)
    raw_flat = raw_product.flatten(start_dim=2)                   # (N,K_out,9)
    C = cg_change_of_basis.to(A_l1)
    coupled = torch.einsum("ab,nkb->nka", C, raw_flat)         # 1+3+5
    return {
        0: coupled[:, :, :1].unsqueeze(1),   # (N,path=1,K_out,M=1)
        1: coupled[:, :, 1:4].unsqueeze(1),  # (N,path=1,K_out,M=3)
        2: coupled[:, :, 4:9].unsqueeze(1),  # (N,path=1,K_out,M=5)
    }

def mix_paths_into_message(
    B_by_L, receiver_species, message_weights_by_species_and_path,
):
    """Eq. 11: sum the path axis and preserve (N,K_L,2*L+1)."""
    message_by_L = {}
    for L, B_L in B_by_L.items():
        weight_table = message_weights_by_species_and_path[L].to(B_L)  # (Z,K,P_L)
        node_weights = weight_table[receiver_species].permute(0, 2, 1) # (N,P_L,K)
        if node_weights.shape[1:3] != B_L.shape[1:3]:
            raise ValueError(f"path/channel mismatch for L={L}")
        message_by_L[L] = torch.einsum("npk,npkm->nkm", node_weights, B_L)
    return message_by_L

def update_readout_and_forces(
    h_by_L, message_by_L, positions, scalar_readout_weights,
):
    """Residual update -> atom energies -> total energy and negative gradient."""
    h_next_by_L = {}
    for L in sorted(set(h_by_L) | set(message_by_L)):
        if L in h_by_L and L in message_by_L:
            if h_by_L[L].shape != message_by_L[L].shape:
                raise ValueError(f"residual shape mismatch for L={L}")
            h_next_by_L[L] = h_by_L[L] + message_by_L[L]
        elif L in message_by_L:
            h_next_by_L[L] = message_by_L[L]
        else:
            h_next_by_L[L] = h_by_L[L]
    scalar_channels = h_next_by_L[0].squeeze(-1)                  # (N,K_0)
    readout = scalar_readout_weights.to(scalar_channels)
    atomic_energy = scalar_channels @ readout                     # (N,)
    total_energy = atomic_energy.sum()
    forces = -torch.autograd.grad(
        total_energy, positions, create_graph=True, retain_graph=True
    )[0]
    return h_next_by_L, atomic_energy, total_energy, forces

def run_minimal_mace(positions, species, parameters):
    """Run the complete educational path: graph -> A -> B -> message -> E,F."""
    edge_index = build_cutoff_graph(positions, parameters["r_cut"])
    displacements, distances, directions = build_relative_geometry(positions, edge_index)
    radial_features, harmonics_by_l = build_edge_features(
        distances, directions, parameters["max_l"],
        parameters["radial_modes"], parameters["r_cut"],
    )
    A_by_l = build_first_layer_A(
        edge_index, radial_features, harmonics_by_l, species,
        parameters["radial_maps_by_l"], parameters["species_embedding"],
    )
    B_by_L = build_product_basis(
        A_by_l, correlation_order=2, selected_order_tuple=(1, 1),
        channel_maps_by_l=parameters["channel_maps_by_l"],
        cg_change_of_basis=parameters["cg_change_of_basis"],
    )
    message_by_L = mix_paths_into_message(
        B_by_L, species, parameters["message_weights_by_species_and_path"],
    )
    initial_h0 = parameters["species_embedding"][:, species].T.unsqueeze(-1)
    h_next, atomic_energy, total_energy, forces = update_readout_and_forces(
        {0: initial_h0.to(positions)}, message_by_L, positions,
        parameters["scalar_readout_weights"],
    )
    return {
        "edge_index": edge_index, "displacements": displacements,
        "distances": distances, "directions": directions,
        "radial_features": radial_features, "harmonics_by_l": harmonics_by_l,
        "A_by_l": A_by_l, "B_by_L": B_by_L,
        "message_by_L": message_by_L, "h_next_by_L": h_next,
        "atomic_energy": atomic_energy, "total_energy": total_energy,
        "forces": forces,
    }

## 20. Final symmetry checks

For every operation, compare the same two routes:

- transform input → run operation
- run operation → transform output

For an invariant output, the two values should be equal. For an order-$l$ output, they should differ only by the declared $D^{(l)}(R)$ action. Also test translation and atom relabelling separately: they are different symmetries with different expected output actions.

### Declared scope: proper rotations, not inversion

The notebook develops $SO(3)$ equivariance: every tested rotation has $\det R=+1$. It does not yet store a parity label $p$, so inversion/reflection tests are intentionally out of scope. An $O(3)$ extension must replace each order label $l$ by an irrep label $(l,p)$ and state the parity of spherical harmonics and hidden blocks before testing inversion.

Expected failure modes include mixing the edge and channel axes, using $m$-dependent channel weights, coupling incompatible $L$ blocks, applying the wrong real-basis convention, omitting the receiver scatter-sum, and silently broadcasting a scalar across the wrong axis.

In [ ]:
# FINAL VERIFICATION MATRIX — one complete differentiable methane pass.
verification_dtype = torch.float64
verification_tolerance = 2e-6
demo_species = torch.tensor([1, 0, 0, 0, 0])  # C=1, H=0
demo_parameters = {
    "r_cut": 1.2,
    "max_l": 1,
    "radial_modes": (1, 2, 3),
    "radial_maps_by_l": {
        0: torch.tensor([[0.40, -0.20], [0.10, 0.30], [-0.15, 0.25]], dtype=verification_dtype),
        1: torch.tensor([[0.20, -0.30], [0.35, 0.10], [-0.10, 0.40]], dtype=verification_dtype),
    },
    # Eq. 9's W[k,z_j]: one (K,Z) table, shared across every angular order l.
    "species_embedding": torch.tensor(
        [[0.80, 0.30], [0.20, 0.60]], dtype=verification_dtype
    ),
    "channel_maps_by_l": {
        1: torch.tensor([[0.70, -0.20], [0.10, 0.60]], dtype=verification_dtype),
    },
    "cg_change_of_basis": C_cart,
    "message_weights_by_species_and_path": {
        0: torch.tensor([[[0.40], [0.20]], [[0.30], [-0.50]]], dtype=verification_dtype),
        1: torch.tensor([[[0.15], [-0.10]], [[0.25], [0.35]]], dtype=verification_dtype),
        2: torch.tensor([[[0.20], [0.30]], [[-0.15], [0.45]]], dtype=verification_dtype),
    },
    "scalar_readout_weights": torch.tensor([0.80, -0.35], dtype=verification_dtype),
}

demo_positions = positions.detach().to(verification_dtype).clone().requires_grad_(True)
baseline = run_minimal_mace(demo_positions, demo_species, demo_parameters)

# Shape and graph contracts. r_cut=1.2 keeps the four C-H bonds in both directions.
assert baseline["edge_index"].shape == (2, 8)
assert baseline["radial_features"].shape == (8, 3)
assert baseline["A_by_l"][0].shape == (5, 2, 1)
assert baseline["A_by_l"][1].shape == (5, 2, 3)
assert baseline["B_by_L"][0].shape == (5, 1, 2, 1)
assert baseline["B_by_L"][1].shape == (5, 1, 2, 3)
assert baseline["B_by_L"][2].shape == (5, 1, 2, 5)
assert baseline["message_by_L"][2].shape == (5, 2, 5)
assert baseline["forces"].shape == demo_positions.shape
assert torch.isfinite(baseline["forces"]).all()
assert torch.linalg.vector_norm(baseline["forces"]) > 1e-10

# Input-validation check: a declared edge with zero displacement is invalid.
try:
    build_relative_geometry(
        torch.zeros(2, 3, dtype=verification_dtype), torch.tensor([[0], [1]])
    )
    raise AssertionError("zero-distance edge was accepted")
except ValueError:
    pass

# Neighbour-row permutation: reverse E while keeping every edge feature aligned.
edge_order = torch.arange(baseline["edge_index"].shape[1] - 1, -1, -1)
A_from_reordered_edges = build_first_layer_A(
    baseline["edge_index"][:, edge_order],
    baseline["radial_features"][edge_order],
    {l_order: block[edge_order] for l_order, block in baseline["harmonics_by_l"].items()},
    demo_species, demo_parameters["radial_maps_by_l"],
    demo_parameters["species_embedding"],
)

# Translation route.
translation = torch.tensor([2.0, -1.5, 0.7], dtype=verification_dtype)
translated_positions = (demo_positions.detach() + translation).requires_grad_(True)
translated = run_minimal_mace(translated_positions, demo_species, demo_parameters)

# Proper-rotation route. Use a fresh double-precision z rotation.
angle = math.pi / 5
c_rot, s_rot = math.cos(angle), math.sin(angle)
demo_R = torch.tensor(
    [[c_rot, -s_rot, 0.0], [s_rot, c_rot, 0.0], [0.0, 0.0, 1.0]],
    dtype=verification_dtype,
)
rotated_positions_for_model = (demo_positions.detach() @ demo_R.T).requires_grad_(True)
rotated = run_minimal_mace(rotated_positions_for_model, demo_species, demo_parameters)
D_demo = {l_order: generate_wigner_D(l_order, demo_R) for l_order in [0, 1]}
C_demo = C_cart.to(verification_dtype)
D_coupled_demo = C_demo @ torch.kron(demo_R, demo_R) @ C_demo.T
D_B_demo = {
    0: D_coupled_demo[:1, :1],
    1: D_coupled_demo[1:4, 1:4],
    2: D_coupled_demo[4:9, 4:9],
}

# Atom relabelling route. New row q contains old atom perm[q].
node_permutation = torch.tensor([2, 0, 4, 1, 3])
permuted_positions = demo_positions.detach()[node_permutation].clone().requires_grad_(True)
permuted_species = demo_species[node_permutation]
permuted = run_minimal_mace(permuted_positions, permuted_species, demo_parameters)

def max_abs_relative(actual, expected):
    difference = torch.max(torch.abs(actual - expected)).item()
    scale = max(torch.max(torch.abs(expected)).item(), 1e-14)
    return difference, difference / scale

verification_pairs = {
    "edge-order pooling A[l=1]": (A_from_reordered_edges[1], baseline["A_by_l"][1]),
    "translation displacement": (translated["displacements"], baseline["displacements"]),
    "translation energy": (translated["total_energy"], baseline["total_energy"]),
    "radial rotation invariance": (rotated["radial_features"], baseline["radial_features"]),
    "A[l=1] rotation": (rotated["A_by_l"][1], baseline["A_by_l"][1] @ D_demo[1].T),
    "B[L=2] rotation": (rotated["B_by_L"][2], baseline["B_by_L"][2] @ D_B_demo[2].T),
    "message[L=2] rotation": (rotated["message_by_L"][2], baseline["message_by_L"][2] @ D_B_demo[2].T),
    "rotation energy": (rotated["total_energy"], baseline["total_energy"]),
    "rotation force": (rotated["forces"], baseline["forces"] @ demo_R.T),
    "atom permutation energy": (permuted["atomic_energy"], baseline["atomic_energy"][node_permutation]),
    "atom permutation force": (permuted["forces"], baseline["forces"][node_permutation]),
    "total permutation invariance": (permuted["total_energy"], baseline["total_energy"]),
}
verification_rows = []
for check_name, (actual, expected) in verification_pairs.items():
    absolute_error, relative_error = max_abs_relative(actual, expected)
    verification_rows.append((check_name, absolute_error, relative_error))
    assert absolute_error <= verification_tolerance, (check_name, absolute_error)

# Symmetric nu=2 self-product: the antisymmetric L=1 sector must vanish.
forbidden_L1_error = torch.max(torch.abs(baseline["B_by_L"][1])).item()
assert forbidden_L1_error <= verification_tolerance
verification_rows.append(("forbidden nu=2 L=1 sector", forbidden_L1_error, forbidden_L1_error))

print(f"dtype={verification_dtype}, absolute tolerance={verification_tolerance:.1e}")
print(f"{'check':<34} | {'max abs':>10} | {'max relative':>12}")
print("-" * 64)
for check_name, absolute_error, relative_error in verification_rows:
    print(f"{check_name:<34} | {absolute_error:10.2e} | {relative_error:12.2e}")

# Intentional visual: which symmetry contract has the largest numerical residue?
plot_names = [row[0] for row in verification_rows]
plot_errors = [max(row[1], 1e-18) for row in verification_rows]
plt.figure(figsize=(9, 6))
plt.barh(plot_names, plot_errors, color="slateblue")
plt.axvline(verification_tolerance, color="crimson", linestyle="--", label="tolerance")
plt.xscale("log")
plt.xlabel("Maximum absolute error (log scale)")
plt.title("Question: which symmetry check has the largest numerical residue?")
plt.legend()
plt.tight_layout()
plt.show()
print("Takeaway: every implemented contract lies to the left of the declared tolerance.")

# Conclusion: the MACE pattern in one sentence

MACE turns relative atomic geometry into typed rotational features, builds higher-body interactions by tensor products, projects those products back into irreducible blocks, and learns only scalar mixtures that cannot disturb how the blocks rotate.

That sentence is compact because the notebook built each noun before asking you to read the paper equations. A direction became spherical harmonics $Y_m^{(l)}$; the angular order $l$ selected a block with $2l+1$ components indexed by $m$; a Wigner matrix $D^{(l)}(R)$ described how that block responds to a rotation $R$. Tensor products then created interactions, and Clebsch–Gordan coefficients reorganized their components into legal output orders $L$ with components $M$.

The MACE-specific compression happens in three steps. Equation 9 pools edge messages into one-centre features $A_{i,klm}$. Equation 10 multiplies $\nu$ such features, so correlation order $\nu$ corresponds to body order $\nu+1$ after counting the centre atom, and the coupling-path label $\eta_\nu$ records how intermediate angular orders were combined. Equation 11 mixes compatible paths and feature channels with scalar learned weights while leaving the output component $M$ untouched.

The final executable block follows exactly that route for the smallest useful case: $A^{(0)}$ and $A^{(1)}$, then the $\nu=2$ coupling $(l_1,l_2)=(1,1)\rightarrow L\in\{0,1,2\}$, then receiver-species path mixing, scalar readout, total energy, and forces. Its tests ask the questions that matter: do distances ignore rotations, do order-$L$ blocks transform with $D^{(L)}$, does atom relabelling merely relabel node outputs, is energy invariant, and do forces rotate as vectors?

> **Final takeaway:** equivariance is not a property to hope the network learns. It is an interface contract carried by every block, axis, and allowed linear map.

# Pass 3 session log

## 2026-08-25 — foundations through a minimal MACE pass

**Objective and scope:** Make the notebook readable end to end for an ML engineer new to geometric deep learning, then connect Eqs. 8–11 to one runnable, differentiable educational implementation.

**Reasoning / derivation:** Established rotations and irreducible blocks before paper notation; introduced a per-angular-order storage convention; separated correlation order $\nu$, body order $\nu+1$, angular order $l/L$, component $m/M$, feature channel $k$, and coupling path $\eta_\nu$; then translated each equation into an axis-explicit tensor operation.

**Exact cells or artifacts completed:** Foundations labs; radial basis and cutoff discussion; the Eq. 8 one-edge ledger; staged Eq. 9, Eq. 10, and Eq. 11 labs; implementation ladder; directed-edge convention; minimal graph-to-energy-and-forces pipeline; final symmetry matrix; conclusion. Only this notebook was changed.

**Verification and tolerances:** Run all code cells sequentially in a fresh namespace. The final matrix uses `float64` and an absolute tolerance of `2e-6` for translation, proper-rotation, edge-order, atom-permutation, forbidden-sector, energy, and force checks.

**Intentional boundary:** This is an $SO(3)$, one-layer educational implementation. General $\nu>2$ path enumeration, parity-labelled $O(3)$ irreps, batching, trainable modules, and dataset training belong in a production implementation; they are not implied by hidden notebook code.

**Next action:** Restart the kernel and run all cells. Read the printed verification matrix as a set of architectural contracts, then replace one fixed scalar map at a time with a trainable module without changing any shape or symmetry test.